In [1]:
# ============================================================
# Cell 0 — Cài package
# ============================================================
%uv pip install --no-cache-dir torch==2.8.0 numpy scipy pandas matplotlib tqdm scikit-learn statsmodels tabulate
# Times New Roman không có sẵn trên nhiều máy Linux; Liberation Serif tương thích số đo với nó.
!apt-get -qq install -y fonts-liberation

Using Python 3.12.6 environment at: /usr/local
Audited 9 packages in 106ms
Note: you may need to restart the kernel to use updated packages.
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package fonts-liberation.
(Reading database ... 33321 files and directories currently installed.)
Preparing to unpack .../fonts-liberation_1%3a1.07.4-11_all.deb ...
Unpacking fonts-liberation (1:1.07.4-11) ...
Setting up fonts-liberation (1:1.07.4-11) ...
Processing triggers for fontconfig (2.14.1-4) ...


# Study 2a — Ngôn ngữ tổng hợp: hourglass có điều khiển (P1, P2, P3, S2–S6)

Theo mục 5 / Study 2a của *"Principle of Least Effort in Depth"* và kế hoạch chạy LE-D (≈50 A100-giờ cho toàn lưới).

**Ý tưởng.** Giả định (G) của LE-D (nội dung độc lập với ngôn ngữ, mỗi ngôn ngữ dịch đơn ánh, văn bản lộ ngôn ngữ) chỉ đúng *theo cách xây dựng* trên ngôn ngữ tự tạo từ một "chương trình ngữ nghĩa" duy nhất. Notebook này (1) sinh các ngôn ngữ tổng hợp với luật rõ ràng và **kiểm tra (G) bằng parser nghịch**, (2) huấn luyện mô hình tí hon (K = 8 tầng) ở bốn họ (decoder, encoder-MLM, MoE, diffusion) với ba tác vụ (LM / SEM / SURF), (3) đo D_k, G_k, CKA, A_k (Procrustes-CV), U_k (usable information) theo tầng, và (4) kiểm định các giả thuyết P1, P2, P3, S2, S3, S4, S5, S6 bằng chính số đo đó.

**Cấu trúc notebook**

| Phần | Nội dung | Cần GPU |
|---|---|---|
| A | config, đường dẫn, kích thước lưới | không |
| S | **bộ sinh ngôn ngữ tổng hợp**: nội dung C (PCFG), ngôn ngữ (từ vựng, hình thái, trật tự từ), renderer, parser nghịch, tokenizer BPE, dựng "thế giới" (corpus + tập đo) | không |
| T | **testcase cho bộ sinh**: luật, thống kê, round-trip, mơ hồ, tokenizer (cổng G0) | không |
| B | mô hình 4 họ, tác vụ, huấn luyện, trích biểu diễn, số đo M1–M4, logit-lens | có |
| C | định nghĩa các lưới thí nghiệm, chạy (có resume), cổng G2 | có |
| D | đặc trưng waist, kiểm định giả thuyết, hình PDF, báo cáo kết quả (chỉ đọc `STUDY_DIR`) | không |

**Lưu trữ (`/mnt/cl/study2a`)**
```
worlds/{world_id}.npz + .json           # corpus tokenized, tập đo, tokenizer, fertility (dùng chung cho mọi family/task/d cùng thế giới)
runs/{run_id}/meta.json                 # cấu hình, loss, fertility, thời gian, trạng thái
runs/{run_id}/{init|trained|t0.25|...}.npz   # số đo theo tầng (G, D, CKA, A, U, g_content, G_rank ...)
runs/{run_id}/twins.npz, lens.npz       # A_indep / CKA_indep của cặp sinh đôi; khối lượng logit-lens (pivot)
analysis/*.csv|json|md                  # bảng đặc trưng, bảng kiểm định, report
figures/*.pdf                           # hình
```
`run_id` = `{grid}__{family}_{level}_{task}_d{d}_L{L}_ov{overlap}_ls{len_scale}_sh{share}_s{seed}`.

**Chạy lại / resume.** Mỗi run đã có `meta.json` với `status = ok` sẽ bị bỏ qua. Chọn lưới bằng `RUN_GRIDS`; chạy lại từ đầu: `FORCE_RERUN = True`. Ở kernel mới chỉ cần chạy Cell A1 rồi các cell Phần D (D1–D7) để vẽ lại hình và báo cáo; Phần D không phụ thuộc torch hay bộ sinh.

**Thứ tự cổng:** G0 (Phần T, không cần GPU) → `RUN_GRIDS = ["gate_g2"]` (một cấu hình decoder M4/SEM, 3 seed: phải có U₀/log L ≈ 1 và có eo; đo thời gian thực rồi nhân lại ngân sách) → các lưới còn lại. `SMOKE = True` để chạy thử cả đường ống với kích thước cực nhỏ trước.

**Các quyết định thiết kế mà kế hoạch để ngỏ (đã ghi trong comment ở đúng chỗ dùng)**
- d cố định = 192 cho lưới P1/P2, cánh cân bằng, MoE/diffusion, S4, S5; sinh đôi chạy ở d ∈ {64, 128, 256, 384}.
- Nhãn SEM = hạng tần suất của vị ngữ gốc, cắt ở 50 lớp; nhãn SURF = (độ dài từ thứ hai tính theo ký tự, ký tự đầu của nó).
- Tập huấn luyện là một *pool* câu: mỗi nội dung chỉ được viết bằng **một** ngôn ngữ (không có dữ liệu song song, giống π = 0).
- $G_k$ dùng kỳ vọng **chính xác** của khoảng cách khi ghép ngẫu nhiên (công thức đóng) thay cho 1 000 lần xáo; test T3 so khớp hai cách.
- Pivot (S4) đo bằng logit-lens trên decoder: khối lượng xác suất rơi vào token của từng ngôn ngữ ở tầng giữa.

**Lưu ý.** Mọi kết luận chỉ "ở quy mô nhỏ, trong ngôn ngữ tổng hợp"; kết quả âm chưa bác bỏ LE-D ở quy mô lớn.

## Phần A — Config, đường dẫn, kích thước lưới

In [ ]:
# ============================================================
# Cell A1 — Config & đường dẫn lưu trữ
# ============================================================
import os, gc, json, time, math, re, itertools, random, bisect, heapq, warnings, collections, traceback
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np
from tqdm.auto import tqdm

SMOKE = False      # True: chạy thử cực nhỏ (vài phút, lưu ở study2a_smoke/) để kiểm tra đường ống trước khi chạy thật
ROOT         = os.environ.get("LED_ROOT", os.getcwd())
STUDY_DIR    = f"{ROOT}/study2a" + ("_smoke" if SMOKE else "")
WORLD_DIR    = f"{STUDY_DIR}/worlds"
RUN_DIR      = f"{STUDY_DIR}/runs"
ANALYSIS_DIR = f"{STUDY_DIR}/analysis"
FIG_DIR      = f"{STUDY_DIR}/figures"
for _d in (WORLD_DIR, RUN_DIR, ANALYSIS_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

def _S(full, small):
    return small if SMOKE else full

SEED = 42

# ---- Ngôn ngữ tổng hợp / dữ liệu ----
N_POOL       = _S(400_000, 12_000)   # số câu trong pool huấn luyện (mỗi câu = một nội dung mới, một ngôn ngữ)
N_EVAL       = _S(3000, 300)         # nội dung giữ ra để đo (x L ngôn ngữ)
N_PROBE      = _S(2000, 300)         # tập dò S3: danh từ chủ ngữ lấy đều theo hạng
BPE_VOCAB    = _S(8000, 1200)        # số ký hiệu BPE chung (ký tự + merge)
BPE_MAX_TYPES = _S(80_000, 20_000)   # số loại từ (theo tần suất) dùng để huấn luyện BPE
MAX_LEN      = 64                    # chiều dài chuỗi (token); câu <= MAX_SENT_TOK + 2 (bos/eos), chừa chỗ cho hậu tố nhãn
MAX_SENT_TOK = 60
MIN_WORDS    = 3                     # R27: câu ngắn hơn bị loại (giảm mơ hồ chéo ngôn ngữ ở giao diện cân bằng)
MAX_DEPTH    = 4                     # R25
HELDOUT_PCT  = 5                     # R28: % tổ hợp (vị ngữ, danh từ arg0, danh từ arg1) bị giữ ra
N_SEM_CLS    = 50

# ---- Mô hình / huấn luyện (Study 2a, kế hoạch mục 4) ----
K_LAYERS = 8
D_HEAD   = 32
D_LIST   = [64, 96, 128, 192, 256, 384]     # B ~ 0,4–14M tham số không-embedding
D_MAIN   = 192                              # d dùng cho các lưới không quét capacity
D_TWIN   = [64, 128, 256, 384]              # d có sinh đôi đơn ngữ (P3)

# ---- Batch / số bước ----
# Cấu hình tham chiếu của kế hoạch: BATCH_REF chuỗi/bước x STEPS_REF bước (~7,7 epoch trên pool 400K câu).
# Tăng BATCH => STEPS và WARMUP tự giảm theo tỉ lệ để TỔNG SỐ CHUỖI ĐÃ XEM giữ nguyên;
# lr đỉnh được nhân (BATCH / BATCH_REF) ** LR_BATCH_EXP  (0.5 = căn bậc hai; 0 = giữ nguyên lr; 1 = tuyến tính).
# Đặt BATCH = BATCH_REF để tái lập đúng cấu hình cũ (256 x 12000 bước, lr gốc).
BATCH_REF  = _S(256, 64)
STEPS_REF  = _S(12000, 150)
WARMUP_REF = _S(500, 15)
BATCH        = _S(4096, 64)       # A100 80GB: thử 2048 / 4096 nếu còn dư VRAM (logits fp32 là phần tốn nhất, ~2,4 GB / 1024 chuỗi)
LR_BATCH_EXP = 0.5
STEPS_SCALE  = BATCH_REF / BATCH
STEPS    = max(1, round(STEPS_REF * STEPS_SCALE))
WARMUP   = max(1, round(WARMUP_REF * STEPS_SCALE))
LR_SCALE = (BATCH / BATCH_REF) ** LR_BATCH_EXP
EVAL_BS  = _S(8192, 128)          # batch cho các forward KHÔNG gradient: trích biểu diễn, logit-lens, eval loss
AUX_FRAC = 0.20          # tỉ lệ chuỗi trong batch có nhãn phụ (SEM / SURF)
AUX_W    = 1.0           # trọng số loss nhãn phụ so với loss LM trung bình
MOE_AUX  = 0.01          # hệ số load-balancing của MoE

# ---- Đo ----
MASK_LEVELS  = [0.0, 0.25, 0.5, 0.75]    # diffusion (S6)
N_MASK_DRAWS = _S(4, 2)
PROBE_SEEDS  = _S(2, 1)
D_PCA        = 64
N_BOOT       = _S(500, 100)
N_RANK_BINS  = 12                         # S3: bin theo floor(log2(hạng)) của danh từ (hạng 1..2500 -> 0..11)

# ---- Điều khiển chạy ----
RUN_GRIDS   = ["gate_g2"]     # xem Cell C1: gate_g2 | smoke | p1p2 | balanced | moe_diff | p3s2 | s4 | s5 | heldout_p4
FORCE_RERUN = False
print(f"[config] STUDY_DIR={STUDY_DIR}  SMOKE={SMOKE}  N_POOL={N_POOL}\n"
      f"[config] BATCH={BATCH} (ref {BATCH_REF})  STEPS={STEPS} (ref {STEPS_REF})  WARMUP={WARMUP}  "
      f"lr_scale={LR_SCALE:.2f}  EVAL_BS={EVAL_BS}  -> {BATCH * STEPS / N_POOL:.1f} epoch")

[config] STUDY_DIR=/mnt/cl/study2a  SMOKE=False  N_POOL=400000
[config] BATCH=4096 (ref 256)  STEPS=750 (ref 12000)  WARMUP=31  lr_scale=4.00  EVAL_BS=4096  -> 7.7 epoch


## Phần S — Bộ sinh ngôn ngữ tổng hợp

Hai tầng tách bạch, đúng như X = g_Λ(C) trong lý thuyết:
1. **Nội dung C** (ngôn ngữ-độc-lập): cây ngữ nghĩa sinh bởi PCFG ≈ 30 luật (R01–R28). Khái niệm lấy mẫu theo Zipf α = 1.
2. **Renderer g_ℓ** (đặc thù ngôn ngữ): từ vựng (khái niệm → chuỗi âm tiết, độ dài ∝ −log tần suất × `len_scale`), trật tự từ (SVO/SOV/VSO), hướng đầu, phụ tố tiền/hậu tố, cách và hòa hợp (luật T01–T10). Mỗi từ ghi lại nút C gốc nên gióng hàng từ chính xác tuyệt đối.

Thang khác biệt giữa các ngôn ngữ: **M0** chỉ từ vựng; **M2** thêm trật tự từ + vị trí phụ tố; **M4** thêm hướng đầu, cách, hòa hợp. Overlap âm vị: **0 %** (mỗi ngôn ngữ một bảng chữ riêng, giao diện lộ ngôn ngữ) hoặc **100 %** (mọi ngôn ngữ dùng chung bảng âm tiết, "cánh cân bằng giao diện", không nhãn ngôn ngữ).

(G) được kiểm bằng `Parser` (parser nghịch theo span, đếm số cây hợp lệ): G2 (đơn ánh) ⇔ đúng một cây và đó là C; G3 (văn bản lộ ngôn ngữ) ⇔ các ngôn ngữ khác không phân tích được chuỗi.

In [3]:
# ============================================================
# Cell S1 — Ngữ nghĩa C: kho khái niệm + văn phạm PCFG
# ============================================================
# ====================== QUY TẮC NỘI DUNG ======================
# Kho khái niệm (5000):
#   R00  id 0..2499 DANH TỪ (N) | 2500..3999 ĐỘNG TỪ (V; 60% bắc cầu val=2, 40% nội động val=1)
#        | 4000..4699 TÍNH TỪ (ADJ) | 4700..4999 TRẠNG TỪ (ADV).
#        Mọi lựa chọn khái niệm lấy mẫu theo Zipf alpha=1 TRONG từng nhóm: P(hạng r) ∝ 1/r.
# Mệnh đề S (cây có kiểu):  ('S', pred, tense, neg, adv, args)   args = (NP,) hoặc (NP, NP)
#   R01  S -> PRED1 NP NP          p = 0.50   (động từ bắc cầu: arg0 = chủ thể, arg1 = đối tượng)
#   R02  S -> PRED0 NP             p = 0.50   (động từ nội động: arg0)
#   R03-R05  thì:      PAST 0.30 | PRES 0.50 | FUT 0.20      (mã 0, 1, 2)
#   R06-R07  phủ định: POS 0.85  | NEG 0.15
#   R08-R09  trạng từ: không 0.70 | một ADV 0.30            (ADV chỉ bổ nghĩa cho động từ)
# Cụm danh từ NP:  ('N', noun, num, def, adjs, ext)
#   R10-R11  số:        SG 0.70 | PL 0.30
#   R12-R13  xác định:  INDEF 0.50 | DEF 0.50
#   R14-R16  số tính từ: 0 (0.55) | 1 (0.30) | 2 (0.15)       (ADJ chỉ bổ nghĩa cho danh từ)
#   R17-R19  mở rộng:   không 0.80 | POSS NP (sở hữu, đệ quy) 0.10 | REL S0 (mệnh đề quan hệ nội động) 0.10
# Từ vựng (Zipf):  R20 N->noun_i | R21 PRED0->verb val1 | R22 PRED1->verb val2 | R23 ADJ->adj_i | R24 ADV->adv_i
# Ràng buộc:
#   R25  độ sâu lồng <= MAX_DEPTH (=4): chạm giới hạn thì R17 bắt buộc "không mở rộng"
#   R26  mệnh đề nằm trong REL luôn là R02 (nội động)
#   R27  câu có < MIN_WORDS từ bị loại và lấy mẫu lại
#   R28  tổ hợp (vị ngữ, danh từ arg0, danh từ arg1) có băm % 100 < HELDOUT_PCT bị GIỮ RA:
#        không bao giờ vào huấn luyện, chỉ dùng để đo (chia theo tổ hợp vị ngữ–đối số)
# =============================================================
N_NOUN, N_VERB, N_ADJ, N_ADV = 2500, 1500, 700, 300
NOUN0, VERB0, ADJ0, ADV0 = 0, 2500, 4000, 4700
N_CONCEPT = 5000
ZIPF_ALPHA = 1.0
P_INTR = 0.50
P_NEG, P_ADV, P_PL, P_DEF = 0.15, 0.30, 0.30, 0.50
P_TENSE = (0.30, 0.50, 0.20)
P_NADJ = (0.55, 0.30, 0.15)
P_POSS, P_REL = 0.10, 0.10

def category(cid):
    if cid < VERB0: return "N"
    if cid < ADJ0: return "V"
    if cid < ADV0: return "ADJ"
    return "ADV"

def verb_valency(cid):
    return 2 if ((cid - VERB0) % 5) < 3 else 1

def _zipf_probs(n):
    w = [1.0 / (r ** ZIPF_ALPHA) for r in range(1, n + 1)]
    tot = sum(w)
    return [x / tot for x in w]

def _cdf(p):
    acc, out = 0.0, []
    for x in p:
        acc += x; out.append(acc)
    out[-1] = 1.0
    return out

POOLS = {"N": list(range(NOUN0, NOUN0 + N_NOUN)),
         "V1": [c for c in range(VERB0, ADJ0) if verb_valency(c) == 1],
         "V2": [c for c in range(VERB0, ADJ0) if verb_valency(c) == 2],
         "ADJ": list(range(ADJ0, ADV0)), "ADV": list(range(ADV0, N_CONCEPT))}
POOL_PROBS = {k: _zipf_probs(len(v)) for k, v in POOLS.items()}
POOL_CDF = {k: _cdf(p) for k, p in POOL_PROBS.items()}

# hạng tần suất trong nhóm lấy mẫu của từng khái niệm (1 = thường gặp nhất); dùng cho độ dài từ và cho S3
CONCEPT_RANK = {}
for _k, _v in POOLS.items():
    for _r, _c in enumerate(_v, 1):
        CONCEPT_RANK[_c] = _r

# hạng tổng thể của động từ theo xác suất xuất hiện làm vị ngữ chính -> nhãn SEM (50 lớp)
_vp = [(P_INTR * POOL_PROBS["V1"][r] if verb_valency(c) == 1 else (1 - P_INTR) * POOL_PROBS["V2"][r], c)
       for k in ("V1", "V2") for r, c in enumerate(POOLS[k])]
VERB_OVERALL_RANK = {c: i + 1 for i, (_, c) in enumerate(sorted(_vp, key=lambda x: (-x[0], x[1])))}

def _pick(rng, pool):
    return POOLS[pool][bisect.bisect_left(POOL_CDF[pool], rng.random())]

def sample_np(rng, depth):
    noun = _pick(rng, "N")
    num = 1 if rng.random() < P_PL else 0                       # R10-R11
    df = 1 if rng.random() < P_DEF else 0                       # R12-R13
    u = rng.random()
    na = 0 if u < P_NADJ[0] else (1 if u < P_NADJ[0] + P_NADJ[1] else 2)   # R14-R16
    adjs = tuple(_pick(rng, "ADJ") for _ in range(na))
    ext = None
    if depth < MAX_DEPTH:                                       # R25
        u = rng.random()
        if u < P_POSS:                                          # R18
            ext = ("POSS", sample_np(rng, depth + 1))
        elif u < P_POSS + P_REL:                                # R19 (+R26)
            ext = ("REL", sample_clause(rng, depth + 1, intransitive=True))
    return ("N", noun, num, df, adjs, ext)

def sample_clause(rng, depth, intransitive=False):
    val = 1 if (intransitive or rng.random() < P_INTR) else 2   # R01 / R02
    pred = _pick(rng, "V1" if val == 1 else "V2")               # R21 / R22
    u = rng.random()
    tense = 0 if u < P_TENSE[0] else (1 if u < P_TENSE[0] + P_TENSE[1] else 2)   # R03-R05
    neg = 1 if rng.random() < P_NEG else 0                      # R06-R07
    adv = _pick(rng, "ADV") if rng.random() < P_ADV else -1     # R08-R09
    args = tuple(sample_np(rng, depth) for _ in range(val))
    return ("S", pred, tense, neg, adv, args)

def np_words(np_):
    n = 1 + len(np_[4])
    if np_[5] is not None:
        n += 1 + (np_words(np_[5][1]) if np_[5][0] == "POSS" else clause_words(np_[5][1]))
    return n

def clause_words(S):
    return 1 + (1 if S[4] >= 0 else 0) + sum(np_words(a) for a in S[5])

def clause_depth(S):
    def nd(np_):
        if np_[5] is None: return 1
        return 1 + (nd(np_[5][1]) if np_[5][0] == "POSS" else clause_depth(np_[5][1]))
    return max(nd(a) for a in S[5])

def is_heldout(C):                                              # R28
    n1 = C[5][1][1] if len(C[5]) == 2 else -1
    h = ((C[1] * 1000003 + C[5][0][1] * 10007 + (n1 + 1) * 101) * 2654435761) % (2 ** 32)
    return (h >> 8) % 100 < HELDOUT_PCT

class ContentSampler:
    """Dòng nội dung tất định theo seed. heldout=False: nội dung cho huấn luyện; heldout=True: nội dung giữ ra để đo."""
    def __init__(self, seed):
        self.rng = random.Random(f"content-{seed}")
    def sample(self, heldout=False):
        while True:
            C = sample_clause(self.rng, 1)
            if clause_words(C) >= MIN_WORDS and is_heldout(C) == heldout:     # R27, R28
                return C
    def sample_probe(self):
        """Tập dò S3: khung 'V0(N ADJ)' với danh từ chủ ngữ lấy LOG-ĐỀU theo hạng (mỗi bát phân log2 có số mẫu gần bằng nhau,
        kể cả đầu bảng tần suất), tổ hợp (vị ngữ, danh từ) thuộc tập giữ ra."""
        while True:
            noun = NOUN0 + min(int(math.exp(self.rng.random() * math.log(N_NOUN))), N_NOUN) - 1
            pred = _pick(self.rng, "V1")
            C = ("S", pred, 1, 0, -1, (("N", noun, 0, 0, (_pick(self.rng, "ADJ"),), None),))
            if is_heldout(C):
                return C
print("[S1] khái niệm:", {k: len(v) for k, v in POOLS.items()}, "| đã định nghĩa PCFG R00-R28")

[S1] khái niệm: {'N': 2500, 'V1': 600, 'V2': 900, 'ADJ': 700, 'ADV': 300} | đã định nghĩa PCFG R00-R28


In [4]:
# ============================================================
# Cell S2 — Ngôn ngữ: bảng âm, từ vựng, hình thái, trật tự từ; renderer g_l
# ============================================================
# ====================== QUY TẮC THỰC HIỆN (renderer) ======================
# Âm vị và từ vựng:
#   T01  Mỗi âm tiết là CV (2 ký tự). Bảng chữ: 8 phụ âm + 5 nguyên âm => 40 âm tiết/ngôn ngữ.
#   T02  40 âm tiết chia hai lớp rời nhau: 9 âm tiết PHỤ TỐ (mang hình vị chức năng) + 31 âm tiết GỐC (tạo gốc từ).
#        => một từ luôn tách được duy nhất thành [phụ tố]* gốc+ [phụ tố]* (parser dựa vào đây).
#   T03  Gốc từ của khái niệm c = chuỗi n âm tiết gốc, n = max(1, round(len_scale*(0.6 + 0.42*ln(hạng_c)))):
#        khái niệm thường gặp -> từ ngắn (Zipf); len_scale điều khiển "token trên từ" (S4, S5).
#        Gốc từ duy nhất trong từng ngôn ngữ; hết chỗ thì n tăng thêm 1 (từ dài hơn danh nghĩa).
#   T04  Overlap 0%: mỗi ngôn ngữ một bảng chữ RIÊNG, rời nhau (giao diện lộ ngôn ngữ ở mọi token).
#        Overlap 100%: mọi ngôn ngữ dùng CÙNG bảng chữ và CÙNG phân lớp âm tiết (gốc/phụ tố); chỉ cách gán
#        gốc từ và phụ tố cho khái niệm là riêng -> cánh cân bằng giao diện (không nhãn ngôn ngữ).
# Hình thái (mỗi đặc trưng đánh dấu bằng đúng 1 âm tiết phụ tố; giá trị không đánh dấu = mặc định):
#   T05  Danh từ: số (PL đánh dấu, SG không) | xác định (DEF đánh dấu) | cách (ACC đánh dấu, chỉ khi has_case).
#   T06  Động từ: thì (PAST, FUT đánh dấu, PRES không) | phủ định (NEG) | hòa hợp (PL đánh dấu khi chủ ngữ số nhiều, chỉ khi has_agr).
#   T07  Vị trí phụ tố: mỗi đặc trưng là tiền tố (pre) hoặc hậu tố (suf); nhiều phụ tố cùng phía xếp theo thứ tự chuẩn
#        num<def<case (danh từ), tense<neg<agr (động từ). Tính từ/trạng từ không biến hình.
#   T08  Hình vị chức năng POSS, REL là một từ riêng gồm đúng 1 âm tiết phụ tố, nằm giữa phụ thuộc và trung tâm.
# Cú pháp:
#   T09  Trật tự từ trong mệnh đề: SVO | SOV | VSO (nội động: SV, SV, VS). Trạng từ dính với động từ thành "đơn vị V".
#        Hướng đầu cho từng cấu trúc (adj, poss, rel, adv): 'I' = phụ thuộc đứng SAU trung tâm, 'F' = đứng TRƯỚC.
#        NP = [tính từ] danh từ [tính từ] rồi mới đến POSS/REL (mở rộng đệ quy bên phải/trái theo hướng đầu).
#   T09b Hòa hợp hướng đầu: poss và rel luôn CÙNG hướng. (Nếu POSS đứng sau trung tâm mà REL đứng trước, mệnh đề quan hệ
#        có thể bám vào hai danh từ khác nhau -> mơ hồ cấu trúc thật; parser phát hiện ~0,4% câu khi để hai hướng độc lập.)
#   T10  Cách: nếu has_case, arg1 của mệnh đề chính mang ACC; mọi NP lồng và arg0 không đánh dấu.
# Thang khác biệt giữa các ngôn ngữ (sample_typology):
#   M0: SVO, mọi phụ tố hậu tố, hướng đầu 'I', không cách, không hòa hợp (chỉ từ vựng khác)
#   M2: + trật tự từ lấy ngẫu nhiên theo ngôn ngữ + vị trí phụ tố (pre/suf) từng đặc trưng
#   M4: + hướng đầu từng cấu trúc + has_case (p=0.5) + has_agr (p=0.5)
# ==========================================================================
LEVELS = ("M0", "M2", "M4")
ALL_FEATS = ["num", "def", "case", "tense", "neg", "agr"]
NOUN_FEATS = ["num", "def", "case"]
VERB_FEATS = ["tense", "neg", "agr"]
AFFIX_ITEMS = [("num", "PL"), ("def", "DEF"), ("case", "ACC"), ("tense", "PAST"), ("tense", "FUT"),
               ("neg", "NEG"), ("agr", "PL"), ("mark", "POSS"), ("mark", "REL")]
N_AFF = len(AFFIX_ITEMS)
N_CONS, N_VOW = 8, 5

def _letter_pool():
    latin = "abcdefghijklmnopqrstuvwxyz"
    greek = "αβγδεζηθικλμνξοπρστυφχψω"
    cyr = "абвгдежзийклмнопрстуфхцчшщыэюя"
    hira = "あいうえおかきくけこさしすせそたちつてとなにぬねのはひふへほまみむめもやゆよらりるれろわ"
    pool = list(latin + greek + cyr + hira)
    assert len(set(pool)) == len(pool)
    return pool

def sample_typology(rng, level):
    t = dict(order="SVO", head=dict(adj="I", poss="I", rel="I", adv="I"),
             pos={f: "suf" for f in ALL_FEATS}, has_case=False, has_agr=False)
    if level in ("M2", "M4"):
        t["order"] = rng.choice(["SVO", "SOV", "VSO"])
        t["pos"] = {f: rng.choice(["pre", "suf"]) for f in ALL_FEATS}
    if level == "M4":
        t["head"] = {c: rng.choice(["I", "F"]) for c in ("adj", "poss", "adv")}
        t["head"]["rel"] = t["head"]["poss"]      # hòa hợp hướng đầu của hai mở rộng NP (xem T09b), tránh mơ hồ bám
        t["has_case"] = rng.random() < 0.5
        t["has_agr"] = rng.random() < 0.5
    return t

def build_lexicon(rng, stem_sylls, len_scale):
    order = sorted(range(N_CONCEPT), key=lambda c: (CONCEPT_RANK[c], c))     # khái niệm thường gặp được gán trước
    lex, used = [None] * N_CONCEPT, set()
    for cid in order:
        n = max(1, int(len_scale * (0.6 + 0.42 * math.log(CONCEPT_RANK[cid])) + 0.5))   # T03
        while True:
            for _ in range(24):
                s = "".join(rng.choice(stem_sylls) for _ in range(n))
                if s not in used:
                    break
            else:
                n += 1
                continue
            break
        used.add(s); lex[cid] = s
    return lex, {s: c for c, s in enumerate(lex)}

class LangSpec:
    def __init__(self, idx, C, V, stem_sylls, aff, typ, lex, len_scale):
        self.idx, self.C, self.V, self.len_scale = idx, list(C), list(V), len_scale
        self.stem_sylls = list(stem_sylls)
        self.stem_set = set(stem_sylls)
        self.aff = aff                                           # (feat, val) -> âm tiết
        self.aff_inv = {s: k for k, s in aff.items()}
        self.order, self.head, self.pos = typ["order"], typ["head"], typ["pos"]
        self.has_case, self.has_agr = typ["has_case"], typ["has_agr"]
        self.lex = lex
        self.stem2cid = {s: c for c, s in enumerate(lex)}
        self.letters = sorted(set(self.C + self.V))
        self._parser = None
    def typology_key(self):
        return (self.order, tuple(sorted(self.head.items())), tuple(sorted(self.pos.items())), self.has_case, self.has_agr)
    def describe(self):
        return (f"L{self.idx}: {self.order}, head={''.join(self.head[c] for c in ('adj','poss','rel','adv'))}, "
                f"pos={''.join(self.pos[f][0] for f in ALL_FEATS)}, case={int(self.has_case)}, agr={int(self.has_agr)}, "
                f"len_scale={self.len_scale:g}")

def make_languages(L, level, overlap, len_scales, seed):
    assert overlap in (0, 100) and level in LEVELS and len(len_scales) == L
    pool = _letter_pool()
    random.Random(f"{seed}-letters").shuffle(pool)
    n_let = N_CONS + N_VOW
    assert overlap == 100 or L * n_let <= len(pool), "không đủ chữ cái cho overlap 0% với L này"
    def syll_split(C, V, rng):
        syl = [c + v for c in C for v in V]
        rng.shuffle(syl)
        return syl[:N_AFF], syl[N_AFF:]                          # T02: 9 phụ tố + 31 gốc
    shared = None
    if overlap == 100:                                           # T04: cùng bảng chữ, cùng phân lớp cho mọi ngôn ngữ
        C, V = pool[:N_CONS], pool[N_CONS:n_let]
        shared = (C, V) + syll_split(C, V, random.Random(f"{seed}-part"))
    specs = []
    for i in range(L):
        if overlap == 100:
            C, V, aff_pool, stem_sylls = shared
        else:
            C = pool[i * n_let: i * n_let + N_CONS]
            V = pool[i * n_let + N_CONS: (i + 1) * n_let]
            aff_pool, stem_sylls = syll_split(C, V, random.Random(f"{seed}-part{i}"))
        r_aff = random.Random(f"{seed}-aff{i}")
        sh = list(aff_pool); r_aff.shuffle(sh)
        aff = {item: sh[j] for j, item in enumerate(AFFIX_ITEMS)}
        typ = sample_typology(random.Random(f"{seed}-typ{i}"), level)
        lex, _ = build_lexicon(random.Random(f"{seed}-lex{i}"), stem_sylls, len_scales[i])
        specs.append(LangSpec(i, C, V, stem_sylls, aff, typ, lex, len_scales[i]))
    return specs

# ---------------- Renderer g_l ----------------
_TENSE_ITEM = {0: ("tense", "PAST"), 2: ("tense", "FUT")}

def _noun_word(spec, cid, num, df, case_marked):
    marks = {"num": ("num", "PL") if num == 1 else None,
             "def": ("def", "DEF") if df == 1 else None,
             "case": ("case", "ACC") if (case_marked and spec.has_case) else None}
    pre = [spec.aff[marks[f]] for f in NOUN_FEATS if marks[f] and spec.pos[f] == "pre"]
    suf = [spec.aff[marks[f]] for f in NOUN_FEATS if marks[f] and spec.pos[f] == "suf"]
    return "".join(pre) + spec.lex[cid] + "".join(suf)

def _verb_word(spec, cid, tense, neg, plural_subj):
    marks = {"tense": _TENSE_ITEM.get(tense),
             "neg": ("neg", "NEG") if neg == 1 else None,
             "agr": ("agr", "PL") if (spec.has_agr and plural_subj) else None}
    pre = [spec.aff[marks[f]] for f in VERB_FEATS if marks[f] and spec.pos[f] == "pre"]
    suf = [spec.aff[marks[f]] for f in VERB_FEATS if marks[f] and spec.pos[f] == "suf"]
    return "".join(pre) + spec.lex[cid] + "".join(suf)

def _render_np(spec, np_, case_marked, tag):
    _, noun, num, df, adjs, ext = np_
    head = [(_noun_word(spec, noun, num, df, case_marked), noun, tag)]
    mods = [(spec.lex[a], a, "ADJ") for a in adjs]
    core = head + mods if spec.head["adj"] == "I" else mods + head
    if ext is None:
        return core
    kind, sub = ext
    if kind == "POSS":
        subw, mk, side = _render_np(spec, sub, False, "N_other"), [(spec.aff[("mark", "POSS")], -1, "FUNC")], spec.head["poss"]
    else:
        subw, mk, side = _render_clause(spec, sub, False), [(spec.aff[("mark", "REL")], -1, "FUNC")], spec.head["rel"]
    return core + mk + subw if side == "I" else subw + mk + core

def _render_clause(spec, S, top):
    _, pred, tense, neg, adv, args = S
    vw = [(_verb_word(spec, pred, tense, neg, args[0][2] == 1), pred, "V")]
    if adv >= 0:
        aw = [(spec.lex[adv], adv, "ADV")]
        vw = vw + aw if spec.head["adv"] == "I" else aw + vw
    subj = _render_np(spec, args[0], False, "N_subj" if top else "N_other")
    if len(args) == 1:
        return vw + subj if spec.order == "VSO" else subj + vw
    obj = _render_np(spec, args[1], True, "N_obj" if top else "N_other")      # T10
    if spec.order == "SVO": return subj + vw + obj
    if spec.order == "SOV": return subj + obj + vw
    return vw + subj + obj

def render_words(spec, C):
    """Danh sách (từ, id khái niệm gốc hoặc -1, vai trò) theo thứ tự bề mặt."""
    return _render_clause(spec, C, True)

def render(spec, C):
    return " ".join(w for w, _, _ in render_words(spec, C))

In [5]:
# ============================================================
# Cell S3 — Parser nghịch (kiểm tra giả định G) + verify
# ============================================================
# Parser nhận một chuỗi từ và trả về TẬP các cây nội dung C có thể sinh ra nó trong ngôn ngữ spec.
#   G2 (đơn ánh):          Parser(spec).parse(render(spec, C)) == {C}   (đúng một cây, chính là C)
#   G3 (lộ ngôn ngữ):      với mọi ngôn ngữ khác, parse(...) rỗng
# Câu "mơ hồ" = có >1 cây trong chính ngôn ngữ đó, hoặc phân tích được bởi ngôn ngữ khác.
# Cài đặt: phân tích hình thái từng từ (tách theo lớp âm tiết) rồi parse theo span với bộ nhớ đệm, dùng đúng các
# mẫu trật tự/hướng đầu của renderer (T05-T10).
_TENSE_CODE = {"PAST": 0, "FUT": 2}

class Parser:
    def __init__(self, spec):
        self.s = spec

    def analyze(self, w):
        """Từ -> ('MARK', tên) | (loại, cid, (đặc trưng...)) | None."""
        s = self.s
        if not w or len(w) % 2:
            return None
        sy = [w[i:i + 2] for i in range(0, len(w), 2)]
        kinds = []
        for x in sy:
            if x in s.stem_set: kinds.append("S")
            elif x in s.aff_inv: kinds.append("A")
            else: return None
        if "S" not in kinds:
            if len(sy) == 1 and s.aff_inv[sy[0]][0] == "mark":
                return ("MARK", s.aff_inv[sy[0]][1])
            return None
        first = kinds.index("S")
        last = len(kinds) - 1 - kinds[::-1].index("S")
        if any(k != "S" for k in kinds[first:last + 1]):
            return None
        cid = s.stem2cid.get("".join(sy[first:last + 1]))
        if cid is None:
            return None
        cat = category(cid)
        if cat in ("ADJ", "ADV"):
            return (cat, cid, ()) if (first == 0 and last == len(sy) - 1) else None
        allowed = NOUN_FEATS if cat == "N" else VERB_FEATS
        feats = {}
        for group, side in ((sy[:first], "pre"), (sy[last + 1:], "suf")):
            prev = -1
            for x in group:
                f, v = s.aff_inv[x]
                if f == "mark" or f not in allowed or s.pos[f] != side:
                    return None
                if f == "case" and not s.has_case: return None
                if f == "agr" and not s.has_agr: return None
                ix = allowed.index(f)
                if ix <= prev or f in feats:
                    return None
                prev = ix; feats[f] = v
        return (cat, cid, tuple(sorted(feats.items())))

    def parse(self, words):
        s = self.s
        n = len(words)
        an = [self.analyze(w) for w in words]
        if n == 0 or any(a is None for a in an):
            return set()
        memo_core, memo_np, memo_cl = {}, {}, {}

        def core(i, j, acc):
            key = (i, j, acc)
            if key in memo_core: return memo_core[key]
            res = []
            if j > i:
                hi = i if s.head["adj"] == "I" else j - 1
                a = an[hi]
                adj_idx = range(i + 1, j) if s.head["adj"] == "I" else range(i, j - 1)
                if a[0] == "N" and all(an[t][0] == "ADJ" for t in adj_idx):
                    f = dict(a[2])
                    if ("case" in f) == (acc and s.has_case):
                        res.append(("N", a[1], 1 if f.get("num") == "PL" else 0, 1 if "def" in f else 0,
                                    tuple(an[t][1] for t in adj_idx), None))
            memo_core[key] = res
            return res

        def np_(i, j, acc):
            key = (i, j, acc)
            if key in memo_np: return memo_np[key]
            res = set(core(i, j, acc))
            for m in range(i, j):
                a = an[m]
                if a[0] != "MARK":
                    continue
                if a[1] == "POSS":
                    cs, ss = ((i, m), (m + 1, j)) if s.head["poss"] == "I" else ((m + 1, j), (i, m))
                    for c in core(cs[0], cs[1], acc):
                        for sub in np_(ss[0], ss[1], False):
                            res.add(c[:5] + (("POSS", sub),))
                else:
                    cs, ss = ((i, m), (m + 1, j)) if s.head["rel"] == "I" else ((m + 1, j), (i, m))
                    for c in core(cs[0], cs[1], acc):
                        for cl in clause_(ss[0], ss[1], 1):
                            res.add(c[:5] + (("REL", cl),))
            memo_np[key] = res
            return res

        def vinfo(a, adv):
            f = dict(a[2])
            return (a[1], _TENSE_CODE.get(f.get("tense"), 1), 1 if "neg" in f else 0, adv, "agr" in f)

        def vunit(a, b):
            if b - a == 1 and an[a][0] == "V":
                return [vinfo(an[a], -1)]
            if b - a == 2:
                vi, ai = (a, a + 1) if s.head["adv"] == "I" else (a + 1, a)
                if an[vi][0] == "V" and an[ai][0] == "ADV":
                    return [vinfo(an[vi], an[ai][1])]
            return []

        def clause_(i, j, val):
            key = (i, j, val)
            if key in memo_cl: return memo_cl[key]
            res = set()
            def mk(vu, subj, obj=None):
                pred, tense, neg, adv, agr = vu
                if verb_valency(pred) != val or agr != (s.has_agr and subj[2] == 1):
                    return
                res.add(("S", pred, tense, neg, adv, (subj,) if obj is None else (subj, obj)))
            if val == 1:
                for m in range(i + 1, j):
                    if s.order == "VSO":
                        for vu in vunit(i, m):
                            for sb in np_(m, j, False): mk(vu, sb)
                    else:
                        for sb in np_(i, m, False):
                            for vu in vunit(m, j): mk(vu, sb)
            else:
                for a in range(i + 1, j):
                    for b in range(a + 1, j):
                        if s.order == "SVO":   vus, sbs, obs = vunit(a, b), np_(i, a, False), np_(b, j, True)
                        elif s.order == "SOV": vus, sbs, obs = vunit(b, j), np_(i, a, False), np_(a, b, True)
                        else:                  vus, sbs, obs = vunit(i, a), np_(a, b, False), np_(b, j, True)
                        for vu in vus:
                            for sb in sbs:
                                for ob in obs: mk(vu, sb, ob)
            memo_cl[key] = res
            return res

        return clause_(0, n, 1) | clause_(0, n, 2)

def get_parser(spec):
    if spec._parser is None:
        spec._parser = Parser(spec)
    return spec._parser

def verify_languages(specs, sampler, n=500, heldout=False):
    """Kiểm (G) trên n nội dung: round-trip đơn ánh (G2), mơ hồ trong ngôn ngữ, và phân tích được bởi ngôn ngữ khác (G3)."""
    L = len(specs)
    st = dict(n=0, roundtrip_fail=0, ambiguous_within=0, parsed_by_other=0, per_lang_fail=[0] * L)
    for _ in tqdm(range(n), desc="kiểm (G) bằng parser", leave=False, disable=(n < 100), dynamic_ncols=True):
        C = sampler.sample(heldout)
        for l, sp in enumerate(specs):
            words = [w for w, _, _ in render_words(sp, C)]
            res = get_parser(sp).parse(words)
            st["n"] += 1
            if res != {C}:
                st["roundtrip_fail"] += 1; st["per_lang_fail"][l] += 1
                if len(res) > 1: st["ambiguous_within"] += 1
            if any(get_parser(o).parse(words) for k, o in enumerate(specs) if k != l):
                st["parsed_by_other"] += 1
    st["rate_roundtrip_fail"] = st["roundtrip_fail"] / max(st["n"], 1)
    st["rate_parsed_by_other"] = st["parsed_by_other"] / max(st["n"], 1)
    return st
print("[S3] parser nghịch sẵn sàng")

[S3] parser nghịch sẵn sàng


In [6]:
# ============================================================
# Cell S4 — Tokenizer BPE chung (viết tay, không phụ thuộc thư viện ngoài)
# ============================================================
# Bố cục id: [0..5] token đặc biệt | [6 .. 6+N_LABEL) token nhãn phụ (SEM/SURF) | [SYM0 ..) ký hiệu BPE (ký tự + merge).
# Mỗi từ được mã hóa ĐỘC LẬP với tiền tố '▁' (merge không bắt qua ranh giới từ); câu = <bos> từ... <eos>.
BOW = "▁"
SPECIALS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sem>", "<surf>"]
PAD, BOS, EOS, MASK, SEM_TOK, SURF_TOK = range(6)
N_LABEL = 1100
LABEL0 = len(SPECIALS)
SYM0 = LABEL0 + N_LABEL

class SharedBPE:
    def __init__(self, base_chars, merges):
        self.base_chars = sorted(base_chars)
        self.merges = [tuple(m) for m in merges]
        self.rank = {m: i for i, m in enumerate(self.merges)}
        syms = [BOW] + self.base_chars
        seen = set(syms)
        for a, b in self.merges:
            if a + b not in seen:
                seen.add(a + b); syms.append(a + b)
        self.symbols = syms
        self.sym2id = {s: SYM0 + i for i, s in enumerate(syms)}
        self.cache = {}

    @property
    def vocab_total(self):
        return SYM0 + len(self.symbols)

    @classmethod
    def train(cls, word_counts, base_chars, n_symbols):
        words = [[BOW] + list(w) for w in word_counts]
        freqs = list(word_counts.values())
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for wi, seq in enumerate(words):
            for p in zip(seq, seq[1:]):
                pair_cnt[p] += freqs[wi]; where[p].add(wi)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        symbols = {BOW} | set(base_chars)
        merges = []
        pbar = tqdm(total=max(0, n_symbols - len(symbols)), desc="BPE merges", leave=False, mininterval=2.0, dynamic_ncols=True)
        while len(symbols) < n_symbols:
            best = None
            while heap:
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c > 0 and c == -negc:
                    best = p; break
                if c > 0:
                    heapq.heappush(heap, (-c, p))
            if best is None or pair_cnt[best] < 2:
                break
            a, b = best
            merges.append(best); symbols.add(a + b); pbar.update(1)
            for wi in list(where[best]):
                seq = words[wi]
                for p in zip(seq, seq[1:]):
                    pair_cnt[p] -= freqs[wi]
                new, i = [], 0
                while i < len(seq):
                    if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
                        new.append(a + b); i += 2
                    else:
                        new.append(seq[i]); i += 1
                words[wi] = new
                for p in zip(new, new[1:]):
                    pair_cnt[p] += freqs[wi]; where[p].add(wi)
                    heapq.heappush(heap, (-pair_cnt[p], p))
            for p in [p for p, c in pair_cnt.items() if c <= 0]:
                del pair_cnt[p]
            pair_cnt.pop(best, None); where.pop(best, None)
        pbar.close()
        return cls(base_chars, merges)

    def encode_word(self, w):
        r = self.cache.get(w)
        if r is not None:
            return r
        seq = [BOW] + list(w)
        while len(seq) > 1:
            best, bi = None, -1
            for i in range(len(seq) - 1):
                rk = self.rank.get((seq[i], seq[i + 1]))
                if rk is not None and (best is None or rk < best):
                    best, bi = rk, i
            if best is None:
                break
            a, b = self.merges[best]
            new, i = [], 0
            while i < len(seq):
                if i < len(seq) - 1 and seq[i] == a and seq[i + 1] == b:
                    new.append(a + b); i += 2
                else:
                    new.append(seq[i]); i += 1
            seq = new
        r = [self.sym2id[s] for s in seq]
        self.cache[w] = r
        return r

    def decode_ids(self, ids):
        inv = {v: k for k, v in self.sym2id.items()}
        return "".join(inv[i] for i in ids if i in inv).replace(BOW, " ").strip()

    def token_language(self, letter_owner):
        """tok_lang[id] = chỉ số ngôn ngữ sở hữu token (mọi ký tự cùng một ngôn ngữ), -1 nếu không xác định (đặc biệt, '▁', chữ dùng chung)."""
        out = np.full(self.vocab_total, -1, np.int16)
        for s, i in self.sym2id.items():
            owners = {letter_owner.get(ch, -1) for ch in s if ch != BOW}
            if len(owners) == 1 and -1 not in owners:
                out[i] = owners.pop()
        return out

    def to_json(self):
        return dict(base_chars=self.base_chars, merges=[list(m) for m in self.merges])

    @classmethod
    def from_json(cls, d):
        return cls(d["base_chars"], d["merges"])

In [7]:
# ============================================================
# Cell S5 — Dựng "thế giới": ngôn ngữ + tokenizer + pool huấn luyện + tập đo (có cache)
# ============================================================
# Một thế giới = (L, level, overlap, len_scales, shares, seed). Mọi (family, task, d) cùng thế giới dùng chung file này.
#  - pool: N_POOL câu; MỖI nội dung được viết bằng đúng MỘT ngôn ngữ (chọn theo `shares`) -> không có dữ liệu song song.
#    Nhãn kèm theo: sem (hạng vị ngữ, cắt 50 lớp), surf (độ dài ký tự x ký tự đầu của từ THỨ HAI).
#  - eval: N_EVAL nội dung giữ ra (R28) x L ngôn ngữ, song song theo cấu trúc; wid = chỉ số từ của từng token (-1: đặc biệt/pad).
#  - probe: N_PROBE khung 'V0(N ADJ)' với danh từ chủ ngữ lấy log-đều theo hạng (S3); subj = chỉ số từ của danh từ chủ ngữ.
def world_cfg(L, level, overlap, len_scale, seed, len_scale_X=None, share_X=None):
    ls = [len_scale] * L
    if len_scale_X is not None:
        ls[0] = len_scale_X                              # ngôn ngữ X = chỉ số 0
    sh = [1.0 / L] * L
    if share_X is not None:
        sh = [share_X] + [(1 - share_X) / (L - 1)] * (L - 1)
    return dict(L=L, level=level, overlap=overlap, len_scales=tuple(ls), shares=tuple(sh), seed=seed)

def world_id(w):
    ls, sh = w["len_scales"], w["shares"]
    lst = f"{ls[0]:g}" if len(set(ls)) == 1 else "-".join(f"{x:g}" for x in ls)
    sht = "eq" if max(sh) - min(sh) < 1e-9 else "-".join(f"{x:.2f}" for x in sh)
    return f"L{w['L']}_{w['level']}_ov{w['overlap']}_ls{lst}_sh{sht}_s{w['seed']}" + ("_smoke" if SMOKE else "")

def _encode_sentence(bpe, words):
    ids, wid = [BOS], [-1]
    for j, w in enumerate(words):
        t = bpe.encode_word(w)
        ids += t; wid += [j] * len(t)
    return ids + [EOS], wid + [-1]

def _render_eval(specs, bpe, contents, probe=False):
    L = len(specs)
    ids_l = [[] for _ in range(L)]; wid_l = [[] for _ in range(L)]; subj = []; keep = []
    for ci, C in enumerate(contents):
        enc = []
        for sp in specs:
            wr = render_words(sp, C)
            ids, wid = _encode_sentence(bpe, [w for w, _, _ in wr])
            enc.append((ids, wid, next((j for j, (_, _, r) in enumerate(wr) if r == "N_subj"), -1)))
        if any(len(e[0]) > MAX_SENT_TOK + 2 for e in enc):
            continue
        for l, (ids, wid, sj) in enumerate(enc):
            ids_l[l].append(ids); wid_l[l].append(wid)
        subj.append(enc[0][2]); keep.append(ci)
    N = len(keep)
    ids_a = np.zeros((L, N, MAX_LEN), np.int16); wid_a = np.full((L, N, MAX_LEN), -1, np.int16)
    for l in range(L):
        for n in range(N):
            k = len(ids_l[l][n])
            ids_a[l, n, :k] = ids_l[l][n]; wid_a[l, n, :k] = wid_l[l][n]
    return ids_a, wid_a, np.array(subj, np.int16), keep

def build_world(wcfg, verbose=True):
    t0 = time.time()
    L, seed = wcfg["L"], wcfg["seed"]
    specs = make_languages(L, wcfg["level"], wcfg["overlap"], list(wcfg["len_scales"]), seed)
    alphabet = sorted({ch for s in specs for ch in s.C + s.V})
    letter_idx = {c: i for i, c in enumerate(alphabet)}
    n_cls_surf = 8 * len(alphabet)
    assert n_cls_surf <= N_LABEL, "tăng N_LABEL"
    owner = {}
    if wcfg["overlap"] == 0:
        for s in specs:
            for ch in s.C + s.V:
                owner[ch] = s.idx
    # ---- pool huấn luyện ----
    sampler = ContentSampler(f"{seed}-train")
    rng_l = random.Random(f"{seed}-langpick")
    cum = np.cumsum(wcfg["shares"])
    word2i, sent_w, langs, sems, surfs = {}, [], [], [], []
    for _ in tqdm(range(N_POOL), desc="sinh pool (render)", leave=False, mininterval=2.0, dynamic_ncols=True):
        C = sampler.sample(False)
        l = min(bisect.bisect_left(cum, rng_l.random() * cum[-1]), L - 1)
        ws = [w for w, _, _ in render_words(specs[l], C)]
        sent_w.append([word2i.setdefault(w, len(word2i)) for w in ws])
        langs.append(l)
        sems.append(min(VERB_OVERALL_RANK[C[1]] - 1, N_SEM_CLS - 1))
        w2 = ws[1]
        surfs.append((min(len(w2) // 2, 8) - 1) * len(alphabet) + letter_idx[w2[0]])
    words_by_i = [None] * len(word2i)
    for w, i in word2i.items():
        words_by_i[i] = w
    cnt = np.bincount(np.concatenate([np.array(s) for s in sent_w]), minlength=len(words_by_i))
    top = np.argsort(-cnt)[:BPE_MAX_TYPES]
    bpe = SharedBPE.train({words_by_i[i]: int(cnt[i]) for i in top}, alphabet, BPE_VOCAB)
    word_tok = [bpe.encode_word(w) for w in tqdm(words_by_i, desc="BPE encode từ", leave=False, mininterval=2.0, dynamic_ncols=True)]
    ids_a = np.zeros((N_POOL, MAX_LEN), np.int16); lens = np.zeros(N_POOL, np.int16)
    keep = np.zeros(N_POOL, bool)
    for n, sw in enumerate(tqdm(sent_w, desc="ghép token cho pool", leave=False, mininterval=2.0, dynamic_ncols=True)):
        toks = [BOS] + [t for wi in sw for t in word_tok[wi]] + [EOS]
        if len(toks) <= MAX_SENT_TOK + 2:
            ids_a[n, :len(toks)] = toks; lens[n] = len(toks); keep[n] = True
    pool = dict(ids=ids_a[keep], len=lens[keep], lang=np.array(langs, np.int8)[keep],
                sem=np.array(sems, np.int16)[keep], surf=np.array(surfs, np.int16)[keep])
    # ---- tập đo ----
    se = ContentSampler(f"{seed}-eval")
    ev_ids, ev_wid, _, _ = _render_eval(specs, bpe, [se.sample(True) for _ in range(N_EVAL)])
    pr_c = ContentSampler(f"{seed}-probe")
    pcont = [pr_c.sample_probe() for _ in range(N_PROBE)]
    pr_ids, pr_wid, pr_subj, pkeep = _render_eval(specs, bpe, pcont)
    pr_rank = np.array([pcont[i][5][0][1] + 1 for i in pkeep], np.int16)
    # fertility (token / từ) theo ngôn ngữ trên tập đo
    fert = []
    for l in range(L):
        nt = ((ev_wid[l] >= 0).sum(1)).astype(float); nw = ev_wid[l].max(1) + 1.0
        fert.append(float(nt.sum() / nw.sum()))
    # kiểm (G) nhanh khi dựng
    ver = verify_languages(specs, ContentSampler(f"{seed}-verify"), n=100 if SMOKE else 200)
    tok_lang = bpe.token_language(owner)
    meta = dict(id=world_id(wcfg), cfg=wcfg, vocab_total=bpe.vocab_total, n_symbols=len(bpe.symbols),
                alphabet=alphabet, n_cls_sem=N_SEM_CLS, n_cls_surf=n_cls_surf, fertility=fert,
                n_pool=int(pool["ids"].shape[0]), verify={k: v for k, v in ver.items() if k != "per_lang_fail"},
                typology=[s.describe() for s in specs], bpe=bpe.to_json(), seconds=round(time.time() - t0, 1))
    arrays = {f"pool_{k}": v for k, v in pool.items()}
    arrays.update(eval_ids=ev_ids, eval_wid=ev_wid, probe_ids=pr_ids, probe_wid=pr_wid, probe_subj=pr_subj,
                  probe_rank=pr_rank, tok_lang=tok_lang)
    base = f"{WORLD_DIR}/{world_id(wcfg)}"
    np.savez_compressed(base + ".tmp.npz", **arrays); os.replace(base + ".tmp.npz", base + ".npz")
    with open(base + ".json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False)
    if verbose:
        print(f"[world] {meta['id']}: pool={meta['n_pool']}  vocab={meta['vocab_total']}  fertility={[round(x, 2) for x in fert]}  "
              f"roundtrip_fail={ver['rate_roundtrip_fail']:.4f}  parsed_by_other={ver['rate_parsed_by_other']:.4f}  ({meta['seconds']}s)")
    return load_world(wcfg)

def load_world(wcfg):
    base = f"{WORLD_DIR}/{world_id(wcfg)}"
    with np.load(base + ".npz") as z:
        W = {k: z[k] for k in z.files}
    with open(base + ".json", encoding="utf-8") as f:
        W["meta"] = json.load(f)
    return W

def get_world(wcfg, verbose=True):
    base = f"{WORLD_DIR}/{world_id(wcfg)}"
    if os.path.exists(base + ".npz") and os.path.exists(base + ".json"):
        return load_world(wcfg)
    return build_world(wcfg, verbose)

## Phần T — Testcase cho bộ sinh ngôn ngữ tổng hợp (cổng G0, không cần GPU)

| Cell | Kiểm tra |
|---|---|
| T1 | ví dụ trực quan: cùng một nội dung C viết ra ở M0 / M2 / M4, overlap 0 % và 100 % |
| T2 | thống kê luật PCFG (R01–R28): tần suất khớp xác suất khai báo, Zipf, độ sâu, độ dài tối thiểu, tập giữ ra rời tập huấn luyện |
| T3 | luật thực hiện (T01–T10): từ vựng duy nhất + độ dài theo Zipf, bảng chữ theo overlap, thang M0 < M2 < M4, đơn ánh bằng va chạm chuỗi |
| T4 | **giả định (G)**: round-trip bằng parser (G2) và tỉ lệ câu mơ hồ / phân tích được bởi ngôn ngữ khác (G3), mọi cấp × overlap × seed; tất định theo seed |
| T5 | số đo: D = 0 / G = 1 cho hai biểu diễn trùng nhau, G ≈ 0 cho độc lập, công thức đóng của $\bar D^\pi$ khớp 1 000 lần xáo; tokenizer BPE |
| T6 | tổng kết PASS/FAIL (dừng lại nếu có FAIL trước khi tốn GPU) |

In [8]:
# ============================================================
# Cell T0 — Tiện ích kiểm thử
# ============================================================
TEST_LOG = []
def check(name, ok, detail=""):
    TEST_LOG.append((name, bool(ok), str(detail)))
    print(("PASS " if ok else "FAIL ") + name + (f"  [{detail}]" if detail != "" else ""))

def walk_nps(S, depth=1):
    """Duyệt mọi NP của một mệnh đề, trả về (np, độ sâu)."""
    for a in S[5]:
        yield from _walk_np(a, depth)

def _walk_np(np_, depth):
    yield np_, depth
    if np_[5] is not None:
        if np_[5][0] == "POSS":
            yield from _walk_np(np_[5][1], depth + 1)
        else:
            yield from walk_nps(np_[5][1], depth + 1)

def walk_clauses(S, depth=1):
    yield S, depth
    for np_, d in walk_nps(S, depth):
        if np_[5] is not None and np_[5][0] == "REL":
            yield np_[5][1], d + 1

def _z(p_hat, p, n):
    return abs(p_hat - p) / math.sqrt(max(p * (1 - p), 1e-12) / n)

In [9]:
# ============================================================
# Cell T1 — Ví dụ trực quan: cùng một nội dung C, ba mức khác biệt, hai kiểu giao diện
# ============================================================
def show_tree(C):
    def np_s(n):
        s = f"N{n[1]}" + ("[PL]" if n[2] else "") + ("[DEF]" if n[3] else "") + "".join(f" +ADJ{a}" for a in n[4])
        if n[5]:
            s += f" +{n[5][0]}(" + (np_s(n[5][1]) if n[5][0] == "POSS" else cl_s(n[5][1])) + ")"
        return s
    def cl_s(S):
        t = ["PAST", "PRES", "FUT"][S[2]]
        return f"V{S[1]}[{t}{',NEG' if S[3] else ''}]" + (f" +ADV{S[4]}" if S[4] >= 0 else "") + " <- " + ", ".join(np_s(a) for a in S[5])
    return cl_s(C)

_demo = ContentSampler("demo")
_C_demo = next(c for c in (_demo.sample(False) for _ in range(500)) if clause_words(c) >= 5 and len(c[5]) == 2)
print("Nội dung C:", show_tree(_C_demo), "\n")
for _ov in (0, 100):
    for _lev in LEVELS:
        _sp = make_languages(3, _lev, _ov, [1.0] * 3, seed=7)
        print(f"--- overlap {_ov}%, mức {_lev} ---")
        for s in _sp:
            print(f"  {s.describe():78s} | {render(s, _C_demo)}")

Nội dung C: V2971[FUT] +ADV4710 <- N23[PL], N510 +ADJ4016 

--- overlap 0%, mức M0 ---
  L0: SVO, head=IIII, pos=ssssss, case=0, agr=0, len_scale=1                     | へчычыε へくyпもこのч χくへп もこへくへく yくyε
  L1: SVO, head=IIII, pos=ssssss, case=0, agr=0, len_scale=1                     | kすtхβc てjkрとcてす とcたр wすてckす てрэす
  L2: SVO, head=IIII, pos=ssssss, case=0, agr=0, len_scale=1                     | γяれnσя γnγюzяよh γnzя znzюγю γnれh
--- overlap 0%, mức M2 ---
  L0: VSO, head=IIII, pos=spssss, case=0, agr=0, len_scale=1                     | へくyпもこのч χくへп へчычыε もこへくへく yくyε
  L1: SVO, head=IIII, pos=ssssss, case=0, agr=0, len_scale=1                     | kすtхβc てjkрとcてす とcたр wすてckす てрэす
  L2: SOV, head=IIII, pos=sspsss, case=0, agr=0, len_scale=1                     | γяれnσя znzюγю γnれh γnγюzяよh γnzя
--- overlap 0%, mức M4 ---
  L0: VSO, head=IFFF, pos=spssss, case=1, agr=0, len_scale=1                     | χくへп へくyпもこのч へчычыε もこへくへくχч yくyε
  L1: SVO, head=IIIF, pos=ssssss, case=0, agr

In [10]:
# ============================================================
# Cell T2 — Thống kê luật PCFG (R01–R28)
# ============================================================
_rng = random.Random("t2-rules")
_N = 30000
_raw = [sample_clause(_rng, 1) for _ in range(_N)]         # mẫu thô, TRƯỚC khi áp R27/R28
_cl = [c for S in _raw for c, _d in walk_clauses(S)]
_top = _raw

check("R01/R02 xác suất bắc cầu = 0.50", _z(sum(len(S[5]) == 2 for S in _top) / _N, 0.5, _N) < 4)
for code, p in zip((0, 1, 2), (0.30, 0.50, 0.20)):
    check(f"R03-R05 thì {['PAST','PRES','FUT'][code]} = {p}", _z(sum(S[2] == code for S in _cl) / len(_cl), p, len(_cl)) < 4)
check("R06-R07 phủ định = 0.15", _z(sum(S[3] for S in _cl) / len(_cl), P_NEG, len(_cl)) < 4)
check("R08-R09 trạng từ = 0.30", _z(sum(S[4] >= 0 for S in _cl) / len(_cl), P_ADV, len(_cl)) < 4)

_nps = [(n, d) for S in _top for n, d in walk_nps(S)]
_n = len(_nps)
check("R10-R11 số nhiều = 0.30", _z(sum(n[2] for n, _ in _nps) / _n, P_PL, _n) < 4)
check("R12-R13 xác định = 0.50", _z(sum(n[3] for n, _ in _nps) / _n, P_DEF, _n) < 4)
for k, p in enumerate(P_NADJ):
    check(f"R14-R16 số tính từ = {k} ({p})", _z(sum(len(n[4]) == k for n, _ in _nps) / _n, p, _n) < 4)
_low = [n for n, d in _nps if d < MAX_DEPTH]
check("R18 POSS = 0.10 (khi độ sâu < giới hạn)", _z(sum(n[5] is not None and n[5][0] == "POSS" for n in _low) / len(_low), P_POSS, len(_low)) < 4)
check("R19 REL = 0.10 (khi độ sâu < giới hạn)", _z(sum(n[5] is not None and n[5][0] == "REL" for n in _low) / len(_low), P_REL, len(_low)) < 4)
check("R25 độ sâu <= MAX_DEPTH và không mở rộng ở độ sâu giới hạn",
      all(clause_depth(S) <= MAX_DEPTH for S in _top) and all(n[5] is None for n, d in _nps if d >= MAX_DEPTH),
      f"độ sâu lớn nhất = {max(clause_depth(S) for S in _top)}")
check("R26 mệnh đề trong REL luôn nội động",
      all(len(n[5][1][5]) == 1 for n, _ in _nps if n[5] is not None and n[5][0] == "REL"))

# Zipf trên danh từ trung tâm (R20): độ dốc log-log ≈ -alpha
_cnt = Counter(n[1] for n, _ in _nps)
_r = np.arange(1, 101)
_f = np.array([_cnt.get(NOUN0 + int(i) - 1, 0) for i in _r], float)
_slope = np.polyfit(np.log(_r), np.log(np.maximum(_f, 1)), 1)[0]
check("R20 Zipf: độ dốc log-log của danh từ ≈ -1 (hạng 1-100)", abs(_slope + ZIPF_ALPHA) < 0.15, f"độ dốc = {_slope:.3f}")

# R27 + R28
_s_tr, _s_ev = ContentSampler("t2-train"), ContentSampler("t2-eval")
_tr = [_s_tr.sample(False) for _ in range(20000)]
_ev = [_s_ev.sample(True) for _ in range(2000)]
check("R27 mọi câu >= MIN_WORDS từ", all(clause_words(c) >= MIN_WORDS for c in _tr + _ev), f"MIN_WORDS={MIN_WORDS}")
_combo = lambda c: (c[1], c[5][0][1], c[5][1][1] if len(c[5]) == 2 else -1)
check("R28 không tổ hợp giữ ra nào lọt vào huấn luyện", not ({_combo(c) for c in _tr} & {_combo(c) for c in _ev}))
check("R28 phần trăm tổ hợp bị giữ ra ≈ HELDOUT_PCT", abs(sum(is_heldout(c) for c in _raw) / _N - HELDOUT_PCT / 100) < 0.01,
      f"{sum(is_heldout(c) for c in _raw) / _N:.3%}")
_probe = ContentSampler("t2-probe")
_pc = [_probe.sample_probe() for _ in range(2000)]
_nr = np.array([c[5][0][1] for c in _pc])
check("tập dò S3: danh từ lấy log-đều theo hạng (hạng <= 50 chiếm ≈ 50%) và đều thuộc tập giữ ra",
      abs((_nr < 50).mean() - 0.5) < 0.05 and all(is_heldout(c) for c in _pc) and all(clause_words(c) >= MIN_WORDS for c in _pc))

PASS R01/R02 xác suất bắc cầu = 0.50
PASS R03-R05 thì PAST = 0.3
PASS R03-R05 thì PRES = 0.5
PASS R03-R05 thì FUT = 0.2
PASS R06-R07 phủ định = 0.15
PASS R08-R09 trạng từ = 0.30
PASS R10-R11 số nhiều = 0.30
PASS R12-R13 xác định = 0.50
PASS R14-R16 số tính từ = 0 (0.55)
PASS R14-R16 số tính từ = 1 (0.3)
PASS R14-R16 số tính từ = 2 (0.15)
PASS R18 POSS = 0.10 (khi độ sâu < giới hạn)
PASS R19 REL = 0.10 (khi độ sâu < giới hạn)
PASS R25 độ sâu <= MAX_DEPTH và không mở rộng ở độ sâu giới hạn  [độ sâu lớn nhất = 4]
PASS R26 mệnh đề trong REL luôn nội động
PASS R20 Zipf: độ dốc log-log của danh từ ≈ -1 (hạng 1-100)  [độ dốc = -0.994]
PASS R27 mọi câu >= MIN_WORDS từ  [MIN_WORDS=3]
PASS R28 không tổ hợp giữ ra nào lọt vào huấn luyện
PASS R28 phần trăm tổ hợp bị giữ ra ≈ HELDOUT_PCT  [4.913%]
PASS tập dò S3: danh từ lấy log-đều theo hạng (hạng <= 50 chiếm ≈ 50%) và đều thuộc tập giữ ra


In [11]:
# ============================================================
# Cell T3 — Luật thực hiện (T01–T10)
# ============================================================
from scipy.stats import spearmanr

# T01-T04: bảng chữ, âm tiết, từ vựng
sp0 = make_languages(8, "M4", 0, [1.0] * 8, seed=3)
sp1 = make_languages(8, "M4", 100, [1.0] * 8, seed=3)
check("T04 overlap 0%: bảng chữ rời nhau từng đôi", all(not (set(a.letters) & set(b.letters)) for a, b in itertools.combinations(sp0, 2)))
check("T04 overlap 100%: mọi ngôn ngữ cùng bảng chữ và cùng phân lớp âm tiết",
      all(a.letters == sp1[0].letters and a.stem_set == sp1[0].stem_set for a in sp1))
check("T02 lớp gốc và lớp phụ tố rời nhau; 9 phụ tố + 31 gốc",
      all(not (s.stem_set & set(s.aff_inv)) and len(s.aff_inv) == N_AFF and len(s.stem_set) == N_CONS * N_VOW - N_AFF for s in sp0 + sp1))
check("T03 gốc từ duy nhất trong từng ngôn ngữ (đơn ánh từ vựng)", all(len(set(s.lex)) == N_CONCEPT for s in sp0 + sp1))
check("T04 overlap 100%: cùng khái niệm vẫn khác dạng từ giữa các ngôn ngữ (từ vựng riêng)",
      np.mean([sp1[0].lex[c] != sp1[1].lex[c] for c in range(N_CONCEPT)]) > 0.99)
_rk = np.array([CONCEPT_RANK[c] for c in range(NOUN0, NOUN0 + N_NOUN)])
_ln = np.array([len(sp0[0].lex[c]) // 2 for c in range(NOUN0, NOUN0 + N_NOUN)])
_rho = spearmanr(_rk, _ln).correlation
check("T03 từ dài dần theo hạng (Zipf): Spearman(hạng, độ dài) > 0.6", _rho > 0.6, f"rho = {_rho:.2f}")
_m = {ls: np.mean([len(s.lex[c]) // 2 for c in range(N_CONCEPT)])
      for ls, s in ((ls, make_languages(1, "M0", 0, [ls], 1)[0]) for ls in (0.6, 1.0, 1.6))}
check("T03 len_scale tăng thì độ dài trung bình tăng", _m[0.6] < _m[1.0] < _m[1.6], {k: round(float(v), 2) for k, v in _m.items()})

# thang khác biệt M0 < M2 < M4: khoảng cách typology trung bình giữa các cặp ngôn ngữ
def _typ_vec(s):
    return [s.order] + [s.pos[f] for f in ALL_FEATS] + [s.head[c] for c in ("adj", "poss", "adv")] + [s.has_case, s.has_agr]
def _mean_typ_dist(level, seeds=range(30), L=6):
    d = []
    for sd in seeds:
        sp = make_languages(L, level, 0, [1.0] * L, seed=sd)
        d += [sum(x != y for x, y in zip(_typ_vec(a), _typ_vec(b))) for a, b in itertools.combinations(sp, 2)]
    return float(np.mean(d))
_td = {lv: _mean_typ_dist(lv) for lv in LEVELS}
check("thang khác biệt: khoảng cách typology M0 = 0 < M2 < M4", _td["M0"] == 0 and _td["M0"] < _td["M2"] < _td["M4"], {k: round(v, 2) for k, v in _td.items()})
check("M0: mọi ngôn ngữ SVO, hậu tố, đầu 'I', không cách/hòa hợp",
      all(s.order == "SVO" and set(s.pos.values()) == {"suf"} and set(s.head.values()) == {"I"} and not s.has_case and not s.has_agr
          for s in make_languages(6, "M0", 0, [1.0] * 6, 5)))
check("T09b hòa hợp hướng đầu: poss và rel cùng hướng ở M4", all(s.head["poss"] == s.head["rel"] for s in make_languages(40, "M4", 100, [1.0] * 40, 5)))

# đơn ánh bằng va chạm chuỗi: số chuỗi khác nhau == số nội dung khác nhau (mỗi ngôn ngữ)
_sm = ContentSampler("t3-collide")
_cs = list({_sm.sample(False) for _ in range(30000)})
_ok = True
for s in make_languages(4, "M4", 100, [1.0] * 4, seed=11):
    _ok &= len({render(s, c) for c in _cs}) == len(_cs)
check("G2 (va chạm chuỗi): 30 000 nội dung khác nhau -> 30 000 chuỗi khác nhau ở mỗi ngôn ngữ", _ok)
# bảo toàn thông tin: số từ không phụ thuộc ngôn ngữ và bằng clause_words
check("số từ bề mặt = số nút nội dung (renderer không thêm/bớt từ)",
      all(len(render_words(s, c)) == clause_words(c) for s in sp1[:3] for c in _cs[:2000]))

PASS T04 overlap 0%: bảng chữ rời nhau từng đôi
PASS T04 overlap 100%: mọi ngôn ngữ cùng bảng chữ và cùng phân lớp âm tiết
PASS T02 lớp gốc và lớp phụ tố rời nhau; 9 phụ tố + 31 gốc
PASS T03 gốc từ duy nhất trong từng ngôn ngữ (đơn ánh từ vựng)
PASS T04 overlap 100%: cùng khái niệm vẫn khác dạng từ giữa các ngôn ngữ (từ vựng riêng)
PASS T03 từ dài dần theo hạng (Zipf): Spearman(hạng, độ dài) > 0.6  [rho = 0.86]
PASS T03 len_scale tăng thì độ dài trung bình tăng  [{0.6: 2.8, 1.0: 3.2, 1.6: 5.09}]
PASS thang khác biệt: khoảng cách typology M0 = 0 < M2 < M4  [{'M0': 0.0, 'M2': 3.66, 'M4': 6.2}]
PASS M0: mọi ngôn ngữ SVO, hậu tố, đầu 'I', không cách/hòa hợp
PASS T09b hòa hợp hướng đầu: poss và rel cùng hướng ở M4
PASS G2 (va chạm chuỗi): 30 000 nội dung khác nhau -> 30 000 chuỗi khác nhau ở mỗi ngôn ngữ
PASS số từ bề mặt = số nút nội dung (renderer không thêm/bớt từ)


In [12]:
# ============================================================
# Cell T4 — Giả định (G): round-trip bằng parser, mơ hồ, lộ ngôn ngữ, tất định theo seed
# ============================================================
rows = []
for ov in (0, 100):
    for lev in LEVELS:
        agg = dict(n=0, rt=0, amb=0, other=0)
        for sd in range(3):
            sp = make_languages(4, lev, ov, [1.0] * 4, seed=100 + sd)
            st = verify_languages(sp, ContentSampler(f"t4-{ov}-{lev}-{sd}"), n=600)
            agg["n"] += st["n"]; agg["rt"] += st["roundtrip_fail"]; agg["amb"] += st["ambiguous_within"]; agg["other"] += st["parsed_by_other"]
        rows.append(dict(overlap=ov, level=lev, n=agg["n"], roundtrip_fail=agg["rt"], ambiguous_within=agg["amb"],
                         parsed_by_other=agg["other"], rate_other=agg["other"] / agg["n"]))
import pandas as pd
_t4 = pd.DataFrame(rows)
print(_t4.to_string(index=False))
check("G2: round-trip parser == {C} cho 100% câu, mọi cấp x overlap (0 va chạm cấu trúc)", int(_t4.roundtrip_fail.sum()) == 0,
      f"{int(_t4.roundtrip_fail.sum())}/{int(_t4.n.sum())} lỗi")
check("G3 overlap 0%: không câu nào được ngôn ngữ khác phân tích", int(_t4[_t4.overlap == 0].parsed_by_other.sum()) == 0)
check("G3 overlap 100%: tỉ lệ câu mơ hồ chéo ngôn ngữ < 0.1% (ngưỡng của kế hoạch)", float(_t4[_t4.overlap == 100].rate_other.max()) < 0.001,
      f"lớn nhất = {float(_t4[_t4.overlap == 100].rate_other.max()):.4%}")

# tất định theo seed: cùng seed -> cùng ngôn ngữ và cùng câu; khác seed -> khác
def _fp(seed):
    sp = make_languages(3, "M4", 0, [1.0] * 3, seed=seed)
    cs = ContentSampler(f"fp-{seed}")
    return [render(s, cs.sample(False)) for s in sp for _ in range(200)] + [s.describe() + s.lex[5] for s in sp]
check("tất định: cùng seed cho đúng cùng ngôn ngữ và câu", _fp(1) == _fp(1))
check("khác seed cho thế giới khác", _fp(1) != _fp(2))

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

...

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/600 [00:00<?, ?it/s…

 overlap level    n  roundtrip_fail  ambiguous_within  parsed_by_other  rate_other
       0    M0 7200               0                 0                0    0.000000
       0    M2 7200               0                 0                0    0.000000
       0    M4 7200               0                 0                0    0.000000
     100    M0 7200               0                 0                6    0.000833
     100    M2 7200               0                 0                1    0.000139
     100    M4 7200               0                 0                4    0.000556
PASS G2: round-trip parser == {C} cho 100% câu, mọi cấp x overlap (0 va chạm cấu trúc)  [0/43200 lỗi]
PASS G3 overlap 0%: không câu nào được ngôn ngữ khác phân tích
PASS G3 overlap 100%: tỉ lệ câu mơ hồ chéo ngôn ngữ < 0.1% (ngưỡng của kế hoạch)  [lớn nhất = 0.0833%]
PASS tất định: cùng seed cho đúng cùng ngôn ngữ và câu
PASS khác seed cho thế giới khác


In [14]:
# ============================================================
# Cell T5 — Kiểm thử số đo (G0) và tokenizer
# ============================================================
# ---- bản numpy của M1 để kiểm tính đúng đắn (Phần B dùng cùng công thức bằng torch) ----
def d_std_np(ZA, ZB):
    Z = np.concatenate([ZA, ZB]); mu, sd = Z.mean(0), np.maximum(Z.std(0), 1e-8)
    A, B = (ZA - mu) / sd, (ZB - mu) / sd
    A, B = A - A.mean(0), B - B.mean(0)                      # trừ trung bình từng ngôn ngữ (σ² như paper)
    s2 = (np.square(A).sum(1).mean() + np.square(B).sum(1).mean()) / 2
    return float(np.square(A - B).sum(1).mean() / s2), A, B, s2

def g_exact_np(ZA, ZB):
    D, A, B, s2 = d_std_np(ZA, ZB)
    d_shuf = (np.square(A).sum(1).mean() + np.square(B).sum(1).mean() - 2 * A.mean(0) @ B.mean(0)) / s2   # kỳ vọng chính xác trên hoán vị đều
    return D, 1 - D / d_shuf

_g = np.random.default_rng(0)
_Z = _g.normal(size=(600, 32))
D, G = g_exact_np(_Z, _Z.copy())
check("G0: hai biểu diễn giống hệt -> D = 0, G = 1", abs(D) < 1e-9 and abs(G - 1) < 1e-9)
D, G = g_exact_np(_Z, _g.normal(size=(600, 32)))
check("G0: hai biểu diễn độc lập -> D ≈ 2, G ≈ 0", abs(D - 2) < 0.1 and abs(G) < 0.1, f"D={D:.3f}, G={G:.3f}")
_Zb = _Z @ np.linalg.qr(_g.normal(size=(32, 32)))[0] + 0.3 * _g.normal(size=(600, 32))
D, G = g_exact_np(_Z, _Zb)
_, A_, B_, s2_ = d_std_np(_Z, _Zb)
_mc = np.mean([np.square(A_ - B_[_g.permutation(600)]).sum(1).mean() / s2_ for _ in range(1000)])
check("G0: kỳ vọng chính xác của D_shuffle khớp trung bình 1000 lần xáo (sai số tương đối < 1%)",
      abs((D / (1 - G)) - _mc) / _mc < 0.01, f"đóng={D / (1 - G):.4f}, MC={_mc:.4f}")

# ---- tokenizer BPE ----
_spt = make_languages(4, "M4", 0, [1.0] * 4, seed=21)
_smt = ContentSampler("t5-bpe")
_ws, _lg = [], []
for _i in range(6000):
    _l = _i % 4
    _ws.append([w for w, _, _ in render_words(_spt[_l], _smt.sample(False))]); _lg.append(_l)
_wc = Counter(w for s in _ws for w in s)
_alpha = sorted({ch for s in _spt for ch in s.C + s.V})
_bpe = SharedBPE.train(_wc, _alpha, 700)
check("BPE: số ký hiệu đạt đích (hoặc hết cặp để gộp)", len(_bpe.symbols) <= 700 and len(_bpe.symbols) > len(_alpha) + 100, f"{len(_bpe.symbols)} ký hiệu")
check("BPE: mã hóa rồi giải mã trả lại đúng từ", all(_bpe.decode_ids(_bpe.encode_word(w)) == w for w in list(_wc)[:3000]))
_owner = {ch: s.idx for s in _spt for ch in s.C + s.V}
_tl = _bpe.token_language(_owner)
_enc_lang_ok = all(len({int(_tl[t]) for t in _bpe.encode_word(w) if _tl[t] >= 0}) <= 1 for w in list(_wc)[:2000])
check("BPE + overlap 0%: mọi token của một từ cùng thuộc một ngôn ngữ (token lộ ngôn ngữ)", _enc_lang_ok)
_ids = [t for t in range(SYM0, _bpe.vocab_total) if _tl[t] >= 0]
check("BPE: gần như mọi ký hiệu (trừ '▁') có chủ ngôn ngữ", len(_ids) >= len(_bpe.symbols) - 2, f"{len(_ids)}/{len(_bpe.symbols)}")
def _fert(ls):
    sp = make_languages(2, "M0", 0, [ls] * 2, seed=31)
    sm = ContentSampler(f"t5-f{ls}")
    ws = [[w for w, _, _ in render_words(sp[i % 2], sm.sample(False))] for i in range(3000)]
    b = SharedBPE.train(Counter(w for s in ws for w in s), sorted({c for s in sp for c in s.C + s.V}), 500)
    return float(np.mean([len(b.encode_word(w)) for s in ws for w in s]))
_fr = {ls: _fert(ls) for ls in (0.6, 1.0, 1.6)}
check("fertility (token/từ) tăng theo len_scale (S4, S5)", _fr[0.6] < _fr[1.0] < _fr[1.6], {k: round(float(v), 2) for k, v in _fr.items()})

PASS G0: hai biểu diễn giống hệt -> D = 0, G = 1
PASS G0: hai biểu diễn độc lập -> D ≈ 2, G ≈ 0  [D=1.977, G=0.011]
PASS G0: kỳ vọng chính xác của D_shuffle khớp trung bình 1000 lần xáo (sai số tương đối < 1%)  [đóng=2.0000, MC=2.0003]


BPE merges:   0%|                                                                      | 0/647 [00:00<?, ?it/s…

PASS BPE: số ký hiệu đạt đích (hoặc hết cặp để gộp)  [700 ký hiệu]
PASS BPE: mã hóa rồi giải mã trả lại đúng từ
PASS BPE + overlap 0%: mọi token của một từ cùng thuộc một ngôn ngữ (token lộ ngôn ngữ)
PASS BPE: gần như mọi ký hiệu (trừ '▁') có chủ ngôn ngữ  [699/700]


BPE merges:   0%|                                                                      | 0/473 [00:00<?, ?it/s…

BPE merges:   0%|                                                                      | 0/473 [00:00<?, ?it/s…

BPE merges:   0%|                                                                      | 0/473 [00:00<?, ?it/s…

PASS fertility (token/từ) tăng theo len_scale (S4, S5)  [{0.6: 1.81, 1.0: 1.92, 1.6: 2.77}]


In [13]:
# ============================================================
# Cell T6 — Tổng kết cổng G0 (bộ sinh)
# ============================================================
import pandas as pd
_df = pd.DataFrame(TEST_LOG, columns=["test", "pass", "detail"])
print(_df.assign(status=_df["pass"].map({True: "PASS", False: "FAIL"}))[["status", "test", "detail"]].to_string(index=False))
n_fail = int((~_df["pass"]).sum())
print(f"\nG0 bộ sinh: {len(_df) - n_fail}/{len(_df)} PASS, {n_fail} FAIL")
assert n_fail == 0, "Có test FAIL: sửa bộ sinh trước khi chạy GPU (đừng chạy lưới)."

status                                                                                      test                             detail
  PASS                                                           R01/R02 xác suất bắc cầu = 0.50                                   
  PASS                                                                    R03-R05 thì PAST = 0.3                                   
  PASS                                                                    R03-R05 thì PRES = 0.5                                   
  PASS                                                                     R03-R05 thì FUT = 0.2                                   
  PASS                                                                   R06-R07 phủ định = 0.15                                   
  PASS                                                                   R08-R09 trạng từ = 0.30                                   
  PASS                                                                   R10

## Phần B — Mô hình, tác vụ, huấn luyện, trích biểu diễn, số đo (cần GPU)

- **Bốn họ** (K = 8 tầng, d ∈ {64 … 384}, đầu chú ý ~32 chiều, BPE chung): `decoder` (kiểu Llama: RMSNorm, RoPE, SwiGLU, causal), `encoder` (cùng khối, hai chiều, MLM 15 %), `moe` (decoder, 8 expert top-2, khớp tham số kích hoạt), `diffusion` (hai chiều, che liên tục t ~ U(0,1), trọng số 1/t).
- **Ba tác vụ**: `LM` (chỉ mục tiêu ngôn ngữ), `SEM` (+ 20 % chuỗi dự đoán nhãn ngôn ngữ-độc-lập), `SURF` (+ 20 % chuỗi dự đoán nhãn bề mặt của từ thứ hai).
- **Biểu diễn**: residual stream sau mỗi khối (tầng 0 = embedding; tầng cuối lấy *trước* final norm), gộp trung bình sub-token theo từ rồi trung bình các từ.
- **Số đo**: $D_k$, $\bar G_k$ (kỳ vọng chính xác của khoảng cách ghép ngẫu nhiên), CKA không chệch, Procrustes-CV $A_k$, usable information $U_k$ (probe tuyến tính + MLP, chia nhóm theo nội dung), $G$ theo bin hạng danh từ (S3), logit-lens (pivot, S4).

In [15]:
# ============================================================
# Cell B1 — Torch + mô hình 4 họ
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("device:", DEVICE)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.w

def rope_cache(T, hd, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, hd, 2).float() / hd))
    fr = torch.outer(torch.arange(T).float(), inv)
    return fr.cos(), fr.sin()                                   # [T, hd/2]

def apply_rope(x, cos, sin):                                    # x: [B,T,H,hd]
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h], x[..., h:]
    c = cos[None, :x.shape[1], None, :].to(x.dtype); s = sin[None, :x.shape[1], None, :].to(x.dtype)
    return torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], -1)

class Attn(nn.Module):
    def __init__(self, d, nh, causal):
        super().__init__()
        self.nh, self.hd, self.causal = nh, d // nh, causal
        self.qkv = nn.Linear(d, 3 * d, bias=False); self.o = nn.Linear(d, d, bias=False)
    def forward(self, x, cos, sin, mask):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.nh, self.hd).unbind(2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None if self.causal else mask, is_causal=self.causal)
        return self.o(y.transpose(1, 2).reshape(B, T, -1))

class FFN(nn.Module):                                           # SwiGLU
    def __init__(self, d, ff):
        super().__init__()
        self.w1 = nn.Linear(d, ff, bias=False); self.w3 = nn.Linear(d, ff, bias=False); self.w2 = nn.Linear(ff, d, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MoEFFN(nn.Module):
    """8 expert, top-2, softmax lại trên 2 expert được chọn; expert có d_ff/2 để tham số kích hoạt khớp FFN dày."""
    def __init__(self, d, ff_e, n_exp=8, top_k=2):
        super().__init__()
        self.n_exp, self.top_k = n_exp, top_k
        self.router = nn.Linear(d, n_exp, bias=False)
        self.experts = nn.ModuleList([FFN(d, ff_e) for _ in range(n_exp)])
        self.aux = None
    def forward(self, x):
        B, T, d = x.shape
        xf = x.reshape(-1, d)
        probs = F.softmax(self.router(xf).float(), -1)
        topv, topi = probs.topk(self.top_k, -1)
        topv = topv / topv.sum(-1, keepdim=True)
        out = torch.zeros_like(xf)
        for e in range(self.n_exp):
            rows, slot = (topi == e).nonzero(as_tuple=True)
            if rows.numel() == 0:
                continue
            y = self.experts[e](xf[rows])
            out.index_add_(0, rows, (y * topv[rows, slot].unsqueeze(-1)).to(out.dtype))
        load = F.one_hot(topi, self.n_exp).float().sum(1).mean(0) / self.top_k
        self.aux = self.n_exp * (load * probs.mean(0)).sum()    # loss cân bằng tải (Switch)
        return out.reshape(B, T, d)

class Block(nn.Module):
    def __init__(self, d, nh, ff, moe, causal):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attn(d, nh, causal)
        self.ffn = MoEFFN(d, ff // 2) if moe else FFN(d, ff)
    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.n1(x), cos, sin, mask)
        return x + self.ffn(self.n2(x))

class TinyLM(nn.Module):
    """family: decoder | moe (causal) ; encoder | diffusion (hai chiều). forward -> (logits, hs, hn):
       hs = [residual sau embedding, sau khối 1, ..., sau khối K] (K+1 tensor, cái cuối là TRƯỚC final norm); hn = sau final norm."""
    def __init__(self, family, d, vocab, n_cls=0, K=K_LAYERS):
        super().__init__()
        self.family, self.d = family, d
        self.causal = family in ("decoder", "moe")
        nh = max(1, d // D_HEAD)
        assert d % nh == 0 and (d // nh) % 2 == 0
        ff = ((int(8 * d / 3) + 7) // 8) * 8
        self.emb = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList([Block(d, nh, ff, family == "moe", self.causal) for _ in range(K)])
        self.norm = RMSNorm(d)
        self.cls_head = nn.Linear(d, n_cls) if n_cls else None
        cos, sin = rope_cache(2 * MAX_LEN, d // nh)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        scale = 1.0 / math.sqrt(2 * K)
        for b in self.blocks:
            b.attn.o.weight.data.mul_(scale)
            for m in ([b.ffn] if family != "moe" else list(b.ffn.experts)):
                m.w2.weight.data.mul_(scale)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, ids):
        x = self.emb(ids); hs = [x]
        mask = None if self.causal else (ids != PAD)[:, None, None, :]
        for b in self.blocks:
            x = b(x, self.cos, self.sin, mask); hs.append(x)
        hn = self.norm(x)
        return F.linear(hn, self.emb.weight), hs, hn
    def aux_loss(self):
        return sum(b.ffn.aux for b in self.blocks)
    def n_params_nonemb(self):
        return sum(p.numel() for n, p in self.named_parameters() if not n.startswith("emb") and not n.startswith("cls_head"))

def build_model(family, d, task, W):
    meta = W["meta"]
    n_cls = {"SEM": meta["n_cls_sem"], "SURF": meta["n_cls_surf"]}.get(task, 0) if family == "encoder" else 0
    return TinyLM(family, d, meta["vocab_total"], n_cls).to(DEVICE)

device: cuda


In [16]:
# ============================================================
# Cell B2 — Tác vụ (LM / SEM / SURF) và huấn luyện
# ============================================================
# Chuỗi trong pool: [bos, token..., eos, pad...]; n = số token khác pad.
# LM    : mục tiêu ngôn ngữ của họ (decoder/moe: next-token; encoder: MLM 15% theo 80/10/10; diffusion: che liên tục, loss/t).
# SEM   : thêm AUX_FRAC chuỗi mang nhãn phụ = lớp vị ngữ gốc (50 lớp, độc lập ngôn ngữ).
# SURF  : thêm AUX_FRAC chuỗi mang nhãn phụ = (độ dài ký tự, ký tự đầu) của TỪ THỨ HAI (cần thông tin bề mặt).
#   decoder/moe/diffusion: nhãn là token nối sau câu:  <eos> <sem|surf> <y>   (diffusion: <y> luôn bị che)
#   encoder: đầu tuyến tính trên vị trí 0 (<bos>) dự đoán nhãn
# Tổng loss = loss LM trung bình token + AUX_W * CE nhãn phụ (trung bình theo chuỗi có nhãn).
def pool_to_device(W, lang=None):
    sel = None if lang is None else (W["pool_lang"] == lang)
    out = {}
    for k in ("ids", "len", "sem", "surf"):
        a = W["pool_" + k]
        a = a if sel is None else a[sel]
        out[k] = torch.from_numpy(a.astype(np.int32 if k == "ids" else np.int64)).to(DEVICE)
    return out

def _with_label_suffix(ids, n, aux, y, task):
    full = ids.clone()
    bi = aux.nonzero(as_tuple=True)[0]
    if bi.numel():
        full[bi, n[bi]] = SEM_TOK if task == "SEM" else SURF_TOK
        full[bi, n[bi] + 1] = LABEL0 + y[bi]
    return full, bi

def compute_loss(model, pool, idx, family, task, g, V):
    ids = pool["ids"][idx].long(); n = pool["len"][idx].long()
    B, T = ids.shape
    dev = ids.device
    if task == "LM":
        aux = torch.zeros(B, dtype=torch.bool, device=dev); y = torch.zeros(B, dtype=torch.long, device=dev)
    else:
        aux = torch.rand(B, device=dev, generator=g) < AUX_FRAC
        y = pool["sem" if task == "SEM" else "surf"][idx]
    if family in ("decoder", "moe"):
        full, bi = _with_label_suffix(ids, n, aux, y, task)
        logits, _, _ = model(full[:, :-1])
        tgt = full[:, 1:]
        ce = F.cross_entropy(logits.float().reshape(-1, V), tgt.reshape(-1), reduction="none").view(B, T - 1)
        j = torch.arange(T - 1, device=dev)[None, :]
        is_lab = aux[:, None] & (j == n[:, None]); is_mark = aux[:, None] & (j == (n[:, None] - 1))
        m = (tgt != PAD) & ~is_lab & ~is_mark
        lm = (ce * m).sum() / m.sum().clamp_min(1)
        ax = (ce * is_lab).sum() / is_lab.sum().clamp_min(1)
        return lm + AUX_W * ax, lm.detach()
    if family == "encoder":
        content = ids >= SYM0
        r = torch.rand(ids.shape, device=dev, generator=g)
        sel = content & (r < 0.15)
        none = ~sel.any(1)
        if none.any():
            rows = none.nonzero(as_tuple=True)[0]
            sel[rows, r.masked_fill(~content, 2.0)[rows].argmin(1)] = True
        u = torch.rand(ids.shape, device=dev, generator=g)
        inp = ids.clone()
        inp[sel & (u < 0.8)] = MASK
        m10 = sel & (u >= 0.8) & (u < 0.9)
        inp[m10] = torch.randint(SYM0, V, ids.shape, device=dev, generator=g)[m10]
        logits, _, hn = model(inp)
        lm = F.cross_entropy(logits[sel].float(), ids[sel])
        loss = lm
        if aux.any():
            loss = loss + AUX_W * F.cross_entropy(model.cls_head(hn[:, 0])[aux].float(), y[aux])
        return loss, lm.detach()
    # diffusion (MDLM / LLaDA): che mỗi token nội dung với xác suất t ~ U(0,1), loss CE / t
    full, bi = _with_label_suffix(ids, n, aux, y, task)
    maskable = (ids >= SYM0) | (ids == EOS)
    t = torch.rand(B, device=dev, generator=g).clamp_min(0.05)
    msk = maskable & (torch.rand(ids.shape, device=dev, generator=g) < t[:, None])
    inp = full.clone(); inp[msk] = MASK
    if bi.numel():
        inp[bi, n[bi] + 1] = MASK
    logits, _, _ = model(inp)
    ce = F.cross_entropy(logits.float().reshape(-1, V), full.reshape(-1), reduction="none").view(B, T)
    lm = (ce * msk / t[:, None]).sum() / maskable.sum().clamp_min(1)
    ax = ce[bi, n[bi] + 1].mean() if bi.numel() else ce.new_zeros(())
    return lm + AUX_W * ax, lm.detach()

def train_model(model, family, task, W, steps, seed, lang=None, log=True, desc=None, leave=True):
    pool = pool_to_device(W, lang)
    N, V = pool["ids"].shape[0], W["meta"]["vocab_total"]
    decay = [p for n_, p in model.named_parameters() if p.ndim >= 2 and not n_.startswith("emb")]
    nodecay = [p for n_, p in model.named_parameters() if not (p.ndim >= 2 and not n_.startswith("emb"))]
    lr_peak = (1e-3 if model.d <= 128 else 6e-4) * LR_SCALE      # LR_SCALE = (BATCH/BATCH_REF)**LR_BATCH_EXP (Cell A1)
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": nodecay, "weight_decay": 0.0}],
                            lr=lr_peak, betas=(0.9, 0.95), fused=(DEVICE == "cuda"))
    g = torch.Generator(device=DEVICE); g.manual_seed(5000 + seed)
    warm = max(1, min(WARMUP, steps // 5))
    ema, t0 = None, time.time()
    model.train()
    bar = tqdm(range(steps), desc=desc or f"train {family}/{task}", unit="step", disable=not log, leave=leave,
               mininterval=1.0, dynamic_ncols=True)
    for step in bar:
        lr = lr_peak * (step + 1) / warm if step < warm else \
            lr_peak * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, steps - warm))))
        for gr in opt.param_groups:
            gr["lr"] = lr
        idx = torch.randint(0, N, (BATCH,), device=DEVICE, generator=g)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
            loss, lm = compute_loss(model, pool, idx, family, task, g, V)
            if family == "moe":
                loss = loss + MOE_AUX * model.aux_loss()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % 25 == 0 or step == steps - 1:
            v = float(lm)
            ema = v if ema is None else 0.9 * ema + 0.1 * v
            if log:                                  # chỉ cập nhật postfix khi đã sync GPU (mỗi 25 bước), tránh làm chậm vòng lặp
                bar.set_postfix(lm_loss=f"{ema:.3f}", lr=f"{lr:.1e}", bs=BATCH, refresh=False)
    bar.close()
    model.eval()
    return dict(final_lm_loss=float(ema), steps=steps, batch=BATCH, seconds=round(time.time() - t0, 1))

@torch.no_grad()
def eval_lm_loss(model, W, n=256):
    """Perplexity (nats/token) trên tập giữ ra cho decoder/moe (kiểm tra học được; họ khác trả None)."""
    if model.family not in ("decoder", "moe"):
        return None
    ids = torch.from_numpy(W["eval_ids"][:, :n].reshape(-1, MAX_LEN).astype(np.int64)).to(DEVICE)
    tot, cnt = 0.0, 0
    for s in range(0, ids.shape[0], EVAL_BS):
        b = ids[s:s + EVAL_BS]
        logits, _, _ = model(b[:, :-1])
        tgt = b[:, 1:]; m = tgt != PAD
        tot += float((F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), tgt.reshape(-1), reduction="none") * m.reshape(-1)).sum()); cnt += int(m.sum())
    return tot / max(cnt, 1)

In [17]:
# ============================================================
# Cell B3 — Trích biểu diễn (gộp theo từ)
# ============================================================
def _pool_words(hs, wid):
    """hs: K+1 tensor [B,T,d]; wid: [B,T] (-1 = không thuộc từ). Trả về (câu [K1,B,d], từ [K1,B,W,d]).
    Mỗi từ = trung bình sub-token của nó; câu = trung bình các từ (mỗi từ trọng số như nhau)."""
    H = torch.stack(hs).float()
    W_ = int(wid.max().item()) + 1 if (wid >= 0).any() else 1
    oh = (F.one_hot(wid.clamp_min(0), W_) * (wid >= 0).unsqueeze(-1)).float()
    cnt = oh.sum(1)
    wv = torch.einsum("btw,kbtd->kbwd", oh, H) / cnt.clamp_min(1)[None, :, :, None]
    valid = (cnt > 0).float()
    sent = (wv * valid[None, :, :, None]).sum(2) / valid.sum(1).clamp_min(1)[None, :, None]
    return sent, wv

@torch.no_grad()
def extract_eval(model, ids_all, wid_all, t=None, n_draws=1, pick=None, bs=None, desc=None, progress=True):
    """ids_all, wid_all: np [L,N,T]. Trả về Z [K1,L,N,d] fp16 (câu) và Zp (vector từ `pick` = chỉ số từ mỗi nội dung) nếu có.
    t>0 (diffusion): che ~t token trong từ; cùng vector ngẫu nhiên theo chỉ số nội dung cho mọi ngôn ngữ.
    bs: kích thước batch (mặc định EVAL_BS). Kết quả không phụ thuộc bs (mỗi câu được xử lý độc lập)."""
    model.eval()
    bs = bs or EVAL_BS
    L, N, T = ids_all.shape
    K1 = len(model.blocks) + 1
    Z = torch.empty((K1, L, N, model.d), dtype=torch.float16)
    Zp = torch.empty((K1, L, N, model.d), dtype=torch.float16) if pick is not None else None
    do_mask = t is not None and t > 0
    nd = n_draws if do_mask else 1
    if do_mask:
        U = np.random.default_rng(SEED + int(round(t * 1000))).random((N, nd, T)).astype(np.float32)
    clip = lambda x: torch.nan_to_num(x, nan=0.0, posinf=6e4, neginf=-6e4).clamp(-6e4, 6e4).half().cpu()
    jobs = [(l, s) for l in range(L) for s in range(0, N, bs)]            # (ngôn ngữ, đầu batch)
    for l, s in tqdm(jobs, desc=desc or "trích biểu diễn", unit="batch", leave=False, disable=not progress, dynamic_ncols=True):
        sl = slice(s, min(N, s + bs))
        ids = torch.from_numpy(ids_all[l, sl].astype(np.int64)).to(DEVICE)
        wid = torch.from_numpy(wid_all[l, sl].astype(np.int64)).to(DEVICE)
        acc_s = acc_p = None
        for r in range(nd):
            inp = ids
            if do_mask:
                u = torch.from_numpy(U[sl, r]).to(DEVICE).masked_fill(wid < 0, 2.0)
                m = u < t
                rows = (~m.any(1)).nonzero(as_tuple=True)[0]
                if rows.numel():
                    m[rows, u[rows].argmin(1)] = True
                inp = ids.masked_fill(m, MASK)
            _, hs, _ = model(inp)
            sent, wv = _pool_words(hs, wid)
            acc_s = sent if acc_s is None else acc_s + sent
            if pick is not None:
                pk = torch.from_numpy(pick[sl].astype(np.int64)).to(DEVICE).clamp_min(0)
                pv = wv[:, torch.arange(wv.shape[1], device=DEVICE), pk.clamp_max(wv.shape[2] - 1)]
                acc_p = pv if acc_p is None else acc_p + pv
        Z[:, l, sl] = clip(acc_s / nd)
        if pick is not None:
            Zp[:, l, sl] = clip(acc_p / nd)
    return Z, Zp


In [18]:
# ============================================================
# Cell B4 — Số đo M1–M4 cho một buffer Z [K1, L, N, d]
#   M1: D_k, G_k (kỳ vọng chính xác), g theo từng nội dung (cho bootstrap)    M2: CKA không chệch, Procrustes-CV A_k
#   M4: usable information U_k (probe tuyến tính và MLP, chia nhóm theo nội dung)
#   S3: G theo bin hạng danh từ (rank_alignment)
# ============================================================
METRIC_VERSION = 2   # v2: probe U (đoán ngôn ngữ) giữ lại tâm từng ngôn ngữ. v1 trừ mất tâm -> U ≈ 0 ở mọi tầng (lỗi đo).

def _standardize_keep_lang(Z):
    """Z [L,N,d]: chuẩn hóa từng chiều trên (L,N) nhưng KHÔNG trừ trung bình từng ngôn ngữ.
    Dùng cho probe U: dấu hiệu 'đây là ngôn ngữ nào' nằm chủ yếu ở tâm của mỗi ngôn ngữ; trừ tâm đi thì probe tuyến tính mù."""
    return (Z - Z.mean((0, 1), keepdim=True)) / Z.std(dim=(0, 1), keepdim=True, correction=0).clamp_min(1e-6)

def _standardize(Z):
    """Z [L,N,d]: chuẩn hóa từng chiều trên (L,N), rồi trừ trung bình từng ngôn ngữ (σ² như paper)."""
    Z = (Z - Z.mean((0, 1), keepdim=True)) / Z.std(dim=(0, 1), keepdim=True, correction=0).clamp_min(1e-6)
    return Z - Z.mean(1, keepdim=True)

def _probe_split(Zt, seed):
    L, N, H = Zt.shape
    g = torch.Generator().manual_seed(int(seed))
    perm = torch.randperm(N, generator=g).to(Zt.device)
    te = perm[: N // 2]; rest = perm[N // 2:]
    n_va = max(1, len(rest) // 5); va, trn = rest[:n_va], rest[n_va:]
    flat = lambda c: (Zt[:, c].reshape(-1, H), torch.arange(L, device=Zt.device).repeat_interleave(len(c)))
    (Xtr, ytr), (Xva, yva), (Xte, yte) = flat(trn), flat(va), flat(te)
    mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True).clamp_min(1e-6)
    return [(x - mu) / sd for x in (Xtr, Xva, Xte)], (ytr, yva, yte), L

def usable_info_lin(Zt, seed, lams=(1e-4, 1e-3, 1e-2, 1e-1), iters=60):
    """Zt [L,N,d] đã trừ trung bình theo nội dung. Probe softmax đoán ngôn ngữ; U = log|L| - CE_test (nats)."""
    (Xtr, Xva, Xte), (ytr, yva, yte), L = _probe_split(Zt, seed)
    H, best = Xtr.shape[1], None
    with torch.enable_grad():
        for lam in lams:
            W_ = torch.zeros(H, L, device=Zt.device, requires_grad=True); b_ = torch.zeros(L, device=Zt.device, requires_grad=True)
            opt = torch.optim.LBFGS([W_, b_], lr=1.0, max_iter=iters, line_search_fn="strong_wolfe")
            def closure():
                opt.zero_grad()
                loss = F.cross_entropy(Xtr @ W_ + b_, ytr) + 0.5 * lam * W_.square().sum()
                loss.backward(); return loss
            opt.step(closure)
            with torch.no_grad():
                ce_va = F.cross_entropy(Xva @ W_ + b_, yva).item()
                if best is None or ce_va < best[0]:
                    lo = Xte @ W_ + b_
                    best = (ce_va, F.cross_entropy(lo, yte).item(), (lo.argmax(1) == yte).float().mean().item())
    return math.log(L) - best[1], best[2]

def usable_info_mlp(Zt, seed, hidden=128, steps=150):
    (Xtr, Xva, Xte), (ytr, yva, yte), L = _probe_split(Zt, seed)
    torch.manual_seed(int(seed))
    net = nn.Sequential(nn.Linear(Xtr.shape[1], hidden), nn.GELU(), nn.Linear(hidden, L)).to(Zt.device)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3, weight_decay=1e-3)
    best = None
    with torch.enable_grad():
        for it in range(steps):
            opt.zero_grad(); F.cross_entropy(net(Xtr), ytr).backward(); opt.step()
            if it % 10 == 9:
                with torch.no_grad():
                    cv = F.cross_entropy(net(Xva), yva).item()
                    if best is None or cv < best[0]:
                        best = (cv, F.cross_entropy(net(Xte), yte).item())
    return math.log(L) - best[1]

@torch.no_grad()
def compute_metrics(Z, seed=SEED, probes=True, desc="số đo theo tầng", progress=True):
    K1, L, N, d = Z.shape
    pairs = list(itertools.combinations(range(L), 2)); P = len(pairs)
    nan = lambda *s: np.full(s, np.nan, np.float32)
    G, D, CKA, A = nan(P, K1), nan(P, K1), nan(P, K1), nan(P, K1)
    g_content = nan(K1, N); sig2 = np.zeros(K1, np.float32)
    U_lin, U_mlp, U_acc = nan(K1), nan(K1), nan(K1)
    perm0 = np.random.default_rng(seed).permutation(N)
    tr = torch.tensor(np.sort(perm0[: N // 2]), device=DEVICE); te = torch.tensor(np.sort(perm0[N // 2:]), device=DEVICE)
    d_pca = int(min(D_PCA, len(tr) - 1, d))
    for k in tqdm(range(K1), desc=desc, unit="tầng", leave=False, disable=not progress, dynamic_ncols=True):
        Zraw = Z[k].to(DEVICE, torch.float32)
        Zk = _standardize(Zraw)          # D, G, CKA, A: vẫn trừ tâm từng ngôn ngữ (σ² như paper)
        nrm2 = Zk.square().sum(-1)
        s2 = float(nrm2.mean()); sig2[k] = s2
        if s2 <= 1e-12:
            continue
        Kg = torch.bmm(Zk, Zk.transpose(1, 2)); Kg.diagonal(dim1=1, dim2=2).zero_()
        ksum, krow = Kg.sum((1, 2)), Kg.sum(2)
        def hsic(a, b):
            t1 = (Kg[a] * Kg[b]).sum()
            t2 = ksum[a] * ksum[b] / ((N - 1) * (N - 2))
            t3 = 2.0 / (N - 2) * (krow[a] * krow[b]).sum()
            return (t1 + t2 - t3) / (N * (N - 3))
        self_h = [hsic(l, l) for l in range(L)]
        torch.manual_seed(seed + k)
        Pp = []
        for l in range(L):
            Xtr = Zk[l][tr] - Zk[l][tr].mean(0, keepdim=True)
            _, _, V = torch.pca_lowrank(Xtr, q=d_pca, center=False, niter=3)
            Pl = Zk[l] @ V[:, :d_pca]
            Pp.append(Pl / (Pl[tr].norm() / math.sqrt(len(tr))).clamp_min(1e-8))
        Pp = torch.stack(Pp)
        gsum = torch.zeros(N, device=DEVICE)
        for pi, (a, b) in enumerate(pairs):
            d_real = (Zk[a] - Zk[b]).square().sum(-1)
            d_shuf = nrm2[a].mean() + nrm2[b].mean()          # = E_π ||za_i - zb_π(i)||^2 chính xác (trung bình ngôn ngữ = 0)
            D[pi, k] = float(d_real.mean() / s2)
            G[pi, k] = float(1.0 - d_real.mean() / d_shuf)
            gsum += 1.0 - d_real / d_shuf
            CKA[pi, k] = float(hsic(a, b) / (self_h[a] * self_h[b]).clamp_min(1e-12).sqrt())
            Uu, _, Vh = torch.linalg.svd(Pp[a][tr].T @ Pp[b][tr])
            err = (Pp[b][te] - Pp[a][te] @ (Uu @ Vh)).square().sum() / Pp[b][te].square().sum().clamp_min(1e-12)
            A[pi, k] = float(1.0 - err)
        g_content[k] = (gsum / P).cpu().numpy()
        del Kg, Pp
        if probes:
            Zt = _standardize_keep_lang(Zraw)            # U: giữ tâm từng ngôn ngữ ...
            Zt = Zt - Zt.mean(0, keepdim=True)           # ... nhưng bỏ phần nội dung (trung bình theo nội dung qua các ngôn ngữ)
            r = [usable_info_lin(Zt, seed + si) for si in range(PROBE_SEEDS)]
            U_lin[k] = float(np.mean([x[0] for x in r])); U_acc[k] = float(np.mean([x[1] for x in r]))
            U_mlp[k] = float(np.mean([usable_info_mlp(Zt, seed + si) for si in range(PROBE_SEEDS)]))
    return dict(G=G, D=D, CKA=CKA, A=A, g_content=g_content.astype(np.float16), sig2=sig2,
                U_lin=U_lin, U_mlp=U_mlp, U_acc=U_acc,
                pair_a=np.array([p[0] for p in pairs]), pair_b=np.array([p[1] for p in pairs]))

@torch.no_grad()
def procrustes_between(ZA, ZB, seed=SEED):
    """A_k giữa HAI MÔ HÌNH độc lập (ZA, ZB: [K1,N,d] fp16): quay khớp trên nửa nội dung, chấm trên nửa còn lại.
    Dùng cho A_indep của cặp sinh đôi; cùng thủ tục với A trong compute_metrics."""
    K1, N, d = ZA.shape
    perm0 = np.random.default_rng(seed).permutation(N)
    tr = torch.tensor(np.sort(perm0[: N // 2]), device=DEVICE); te = torch.tensor(np.sort(perm0[N // 2:]), device=DEVICE)
    d_pca = int(min(D_PCA, len(tr) - 1, d))
    out = np.full(K1, np.nan, np.float32); cka = np.full(K1, np.nan, np.float32)
    for k in range(K1):
        Zs = [ (lambda z: (z - z.mean(0)) / z.std(0, correction=0).clamp_min(1e-6))(Z_[k].to(DEVICE).float()) for Z_ in (ZA, ZB)]
        Zs = [z - z.mean(0) for z in Zs]
        torch.manual_seed(seed + k)
        Pp = []
        for z in Zs:
            Xtr = z[tr] - z[tr].mean(0, keepdim=True)
            _, _, V = torch.pca_lowrank(Xtr, q=d_pca, center=False, niter=3)
            Pl = z @ V[:, :d_pca]
            Pp.append(Pl / (Pl[tr].norm() / math.sqrt(len(tr))).clamp_min(1e-8))
        Uu, _, Vh = torch.linalg.svd(Pp[0][tr].T @ Pp[1][tr])
        out[k] = float(1.0 - (Pp[1][te] - Pp[0][te] @ (Uu @ Vh)).square().sum() / Pp[1][te].square().sum().clamp_min(1e-12))
        Ks = [z @ z.T for z in Zs]
        for K_ in Ks: K_.fill_diagonal_(0)
        def hs(Ka, Kb):
            n = Ka.shape[0]
            return ((Ka * Kb).sum() + Ka.sum() * Kb.sum() / ((n - 1) * (n - 2)) - 2.0 / (n - 2) * (Ka.sum(1) * Kb.sum(1)).sum()) / (n * (n - 3))
        cka[k] = float(hs(Ks[0], Ks[1]) / (hs(Ks[0], Ks[0]) * hs(Ks[1], Ks[1])).clamp_min(1e-12).sqrt())
    return out, cka

@torch.no_grad()
def rank_alignment(Zn, ranks):
    """S3: G theo bin hạng danh từ. Zn [K1,L,Np,d]; G_bin = 1 - mean_bin(d_real) / D_shuffle(toàn tập dò), trung bình qua cặp ngôn ngữ."""
    K1, L, Np, d = Zn.shape
    pairs = list(itertools.combinations(range(L), 2))
    b = torch.from_numpy(np.clip(np.floor(np.log2(ranks)).astype(np.int64), 0, N_RANK_BINS - 1)).to(DEVICE)
    cnt = torch.bincount(b, minlength=N_RANK_BINS).float()
    out = np.full((K1, N_RANK_BINS), np.nan, np.float32)
    for k in range(K1):
        Zk = _standardize(Zn[k].to(DEVICE, torch.float32))
        nrm2 = Zk.square().sum(-1)
        if float(nrm2.mean()) <= 1e-12:
            continue
        acc = torch.zeros(N_RANK_BINS, device=DEVICE)
        for a, c in pairs:
            acc.index_add_(0, b, (Zk[a] - Zk[c]).square().sum(-1) / (nrm2[a].mean() + nrm2[c].mean()))
        out[k] = (1.0 - acc / len(pairs) / cnt.clamp_min(1)).cpu().numpy()
        out[k][cnt.cpu().numpy() == 0] = np.nan
    return out, cnt.cpu().numpy()

In [19]:
# ============================================================
# Cell B5 — Logit-lens: ai là "ngôn ngữ trục" (pivot) ở tầng giữa? (S4, chỉ decoder, overlap 0%)
# ============================================================
# Với câu thuộc ngôn ngữ b, áp final norm + ma trận unembedding lên residual của MỖI tầng, rồi tính khối lượng xác suất
# của token kế tiếp rơi vào bộ token của từng ngôn ngữ l (token -> ngôn ngữ nhờ tok_lang, hợp lệ khi overlap 0%).
# M[k, b, l] = khối lượng trung bình (đã chuẩn hóa trên các token có chủ). Ngôn ngữ trục = l có
# pivot_score(l) = mean_{b != l} M[k, b, l] lớn nhất quanh tầng giữa.
@torch.no_grad()
def logit_lens_mass(model, ids_all, tok_lang, n_sub=300, bs=None, progress=True):
    bs = bs or min(EVAL_BS, n_sub)
    L = ids_all.shape[0]
    n_use = min(n_sub, ids_all.shape[1])
    tl = torch.from_numpy(tok_lang.astype(np.int64)).to(DEVICE)
    onehot = torch.zeros(len(tl), L, device=DEVICE)
    own = tl >= 0
    onehot[own.nonzero(as_tuple=True)[0], tl[own]] = 1.0
    K1 = len(model.blocks) + 1
    # tích lũy trên GPU, chỉ chuyển về CPU một lần ở cuối (tránh K1 x số batch lần đồng bộ)
    M = torch.zeros((K1, L, L), dtype=torch.float64, device=DEVICE); C = torch.zeros((L,), dtype=torch.float64, device=DEVICE)
    jobs = [(b, s) for b in range(L) for s in range(0, n_use, bs)]
    for b, s in tqdm(jobs, desc="logit-lens", unit="batch", leave=False, disable=not progress, dynamic_ncols=True):
        ids = torch.from_numpy(ids_all[b, s:min(s + bs, n_use)].astype(np.int64)).to(DEVICE)
        _, hs, _ = model(ids)
        valid = (ids[:, :-1] != PAD) & (ids[:, 1:] >= SYM0)              # vị trí mà token kế tiếp là ký hiệu nội dung
        for k, h in enumerate(hs):
            p = F.softmax(F.linear(model.norm(h[:, :-1]), model.emb.weight).float(), -1)
            mass = p @ onehot
            mass = mass / mass.sum(-1, keepdim=True).clamp_min(1e-9)
            M[k, b] += (mass * valid.unsqueeze(-1)).sum((0, 1)).double()
        C[b] += valid.sum().double()
    M, C = M.cpu().numpy(), C.cpu().numpy()
    return (M / np.maximum(C[None, :, None], 1)).astype(np.float32)


## Phần C — Lưới thí nghiệm, chạy (có resume), cổng G2

| Lưới | Nội dung | Số run |
|---|---|---|
| `gate_g2` | một cấu hình decoder M4/SEM, 3 seed (cổng G2) | 3 |
| `p1p2` | {decoder, encoder} × M{0,2,4} × {LM, SEM, SURF} × 6 seed (P1, P2) | 108 |
| `balanced` | decoder, overlap 100 % × M{0,4} × {LM, SEM} × 6 seed (cánh cân bằng giao diện, A1) | 24 |
| `moe_diff` | {MoE, diffusion} × M4 × 3 tác vụ × 6 seed (P1, S6) | 36 |
| `p3s2` | decoder M4/SEM, 6 giá trị d × L ∈ {2,4,8} × 10 seed; **sinh đôi đơn ngữ** ở d ∈ {64,128,256,384} và seed < 6 (P3, S2, S3) | 180 |
| `s4` | thiết kế 2×2: ngôn ngữ X chiếm 40 % hay đều dữ liệu × fertility X thấp/cao (`len_scale_X`), L = 4, 6 seed + logit-lens (S4) | 24 |
| `s5` | `len_scale` ∈ {0.6, 0.8, 1.0, 1.4, 2.0} × 6 seed (S5) | 30 |
| `heldout_p4` | 24 cấu hình giữ ra (L ∈ {3,6}, d ∈ {80,160,224}, `len_scale` chưa dùng) × 2 seed, có sinh đôi (đầu vào cho Study 5) | 48 |
| `smoke` | bốn họ, d = 64, L = 3: kiểm tra đường ống (dùng với `SMOKE = True`) | 5 |

Một run = {dựng/đọc thế giới → mô hình khởi tạo (đo `init`, null "chưa huấn luyện") → huấn luyện → đo `trained` (+ `t0.25/0.5/0.75` cho diffusion) → (tùy lưới) sinh đôi đơn ngữ, logit-lens}.
Seed điều khiển cả thế giới ngôn ngữ (từ vựng, typology) lẫn khởi tạo mô hình; mọi điều kiện ghép cặp theo seed.

In [20]:
# ============================================================
# Cell C1 — Định nghĩa lưới + run_id
# ============================================================
FAMILIES = ["decoder", "encoder", "moe", "diffusion"]
TASKS = ["LM", "SEM", "SURF"]

def make_cfg(grid, family="decoder", level="M4", task="SEM", d=D_MAIN, L=4, overlap=0, len_scale=1.0,
             len_scale_X=None, share_X=None, seed=0, twins=False, lens=False):
    return dict(grid=grid, family=family, level=level, task=task, d=d, L=L, overlap=overlap, len_scale=len_scale,
                len_scale_X=len_scale_X, share_X=share_X, seed=seed, twins=twins, lens=lens)

def run_id(c):
    ls = f"{c['len_scale']:g}" + (f"X{c['len_scale_X']:g}" if c["len_scale_X"] else "")
    sh = f"{c['share_X']:.2f}" if c["share_X"] else "eq"
    return f"{c['grid']}__{c['family']}_{c['level']}_{c['task']}_d{c['d']}_L{c['L']}_ov{c['overlap']}_ls{ls}_sh{sh}_s{c['seed']}"

def grid_configs(name):
    if name == "gate_g2":
        return [make_cfg("gate_g2", seed=s) for s in range(3)]
    if name == "smoke":
        out = [make_cfg("smoke", family=f, d=64, L=3, task="SEM", twins=(f == "decoder"), lens=(f == "decoder")) for f in FAMILIES]
        return out + [make_cfg("smoke", family="decoder", d=64, L=3, task="LM", overlap=100, level="M2")]
    if name == "p1p2":
        return [make_cfg("p1p2", family=f, level=lv, task=t, seed=s)
                for f in ("decoder", "encoder") for lv in LEVELS for t in TASKS for s in range(6)]
    if name == "balanced":
        return [make_cfg("balanced", overlap=100, level=lv, task=t, seed=s) for lv in ("M0", "M4") for t in ("LM", "SEM") for s in range(6)]
    if name == "moe_diff":
        return [make_cfg("moe_diff", family=f, task=t, seed=s) for f in ("moe", "diffusion") for t in TASKS for s in range(6)]
    if name == "p3s2":
        return [make_cfg("p3s2", d=d, L=L, seed=s, twins=(d in D_TWIN and s < 6)) for d in D_LIST for L in (2, 4, 8) for s in range(10)]
    if name == "s4":
        return [make_cfg("s4", share_X=sx, len_scale_X=lx, seed=s, lens=True) for sx in (None, 0.4) for lx in (0.7, 1.6) for s in range(6)]
    if name == "s5":
        return [make_cfg("s5", len_scale=ls, seed=s) for ls in (0.6, 0.8, 1.0, 1.4, 2.0) for s in range(6)]
    if name == "heldout_p4":
        return [make_cfg("heldout_p4", d=d, L=L, len_scale=ls, seed=s, twins=True)
                for L in (3, 6) for d in (80, 160, 224) for ls in (0.7, 0.9, 1.2, 1.7) for s in range(2)]
    raise KeyError(name)

ALL_GRIDS = ["gate_g2", "smoke", "p1p2", "balanced", "moe_diff", "p3s2", "s4", "s5", "heldout_p4"]
print({g: len(grid_configs(g)) for g in ALL_GRIDS})

{'gate_g2': 3, 'smoke': 5, 'p1p2': 108, 'balanced': 24, 'moe_diff': 36, 'p3s2': 180, 's4': 24, 's5': 30, 'heldout_p4': 48}


In [21]:
# ============================================================
# Cell C2 — Lưu / đọc kết quả, đo một mô hình, sinh đôi, vòng chạy
# ============================================================
def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, path)

def save_npz(rdir, name, arrays):
    os.makedirs(rdir, exist_ok=True)
    tmp = f"{rdir}/{name}.tmp.npz"
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, f"{rdir}/{name}.npz")

def load_npz(rdir, name):
    p = f"{rdir}/{name}.npz"
    if not os.path.exists(p):
        return None
    with np.load(p, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def measure(model, W, rdir, variant, family, masks=False):
    """Đo một mô hình trên tập đo chính + tập dò S3, lưu {variant}.npz. Diffusion: thêm các mức che (S6)."""
    Z, _ = extract_eval(model, W["eval_ids"], W["eval_wid"], desc=f"[{variant}] trích eval")
    arrs = compute_metrics(Z, desc=f"[{variant}] số đo")
    _, Zp = extract_eval(model, W["probe_ids"], W["probe_wid"], pick=W["probe_subj"], desc=f"[{variant}] trích probe S3")
    arrs["G_rank"], arrs["rank_counts"] = rank_alignment(Zp, W["probe_rank"])
    save_npz(rdir, variant, arrs)
    if masks:
        for t in tqdm(MASK_LEVELS[1:], desc=f"[{variant}] mức che", unit="mức", leave=False, dynamic_ncols=True):
            Zt, _ = extract_eval(model, W["eval_ids"], W["eval_wid"], t=t, n_draws=N_MASK_DRAWS, desc=f"[t={t:g}] trích eval")
            save_npz(rdir, f"t{t:g}", compute_metrics(Zt, probes=False, desc=f"[t={t:g}] số đo"))
            del Zt
    return Z

def run_twins(cfg, W, rdir):
    """P3 (Study 2c trên tổng hợp): L mô hình đơn ngữ cùng kiến trúc/tokenizer, mỗi cái T/L token của đúng một ngôn ngữ.
    A_indep[pair, tầng] = Procrustes-CV giữa hai mô hình độc lập; ΔA = A_joint - A_indep (tính ở Phần D)."""
    L = cfg["L"]
    min_steps = max(1, round(_S(500, 30) * STEPS_SCALE))      # sàn số bước cũng co theo BATCH để giữ nguyên tỉ lệ so với mô hình chung
    steps = max(STEPS // L, min_steps)
    Zs, t0, losses = [], time.time(), []
    for l in tqdm(range(L), desc=f"sinh đôi đơn ngữ ({steps} bước/mô hình)", unit="mô hình", leave=False, dynamic_ncols=True):
        torch.manual_seed(2000 + cfg["seed"] * 10 + l)
        tw = build_model(cfg["family"], cfg["d"], cfg["task"], W)
        info = train_model(tw, cfg["family"], cfg["task"], W, steps, cfg["seed"] * 10 + l, lang=l, log=True,
                           desc=f"twin {l + 1}/{L}", leave=False)
        losses.append(info["final_lm_loss"])
        Zl, _ = extract_eval(tw, W["eval_ids"][l:l + 1], W["eval_wid"][l:l + 1], progress=False)
        Zs.append(Zl[:, 0]); del tw
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    pairs = list(itertools.combinations(range(L), 2))
    A_ind = np.full((len(pairs), Zs[0].shape[0]), np.nan, np.float32); C_ind = A_ind.copy()
    for pi, (a, b) in enumerate(tqdm(pairs, desc="Procrustes giữa các cặp sinh đôi", unit="cặp", leave=False, dynamic_ncols=True)):
        A_ind[pi], C_ind[pi] = procrustes_between(Zs[a], Zs[b])
    save_npz(rdir, "twins", dict(A_indep=A_ind, CKA_indep=C_ind, steps=np.array(steps), loss=np.array(losses, np.float32)))
    return round(time.time() - t0, 1)

def run_one(cfg):
    rid = run_id(cfg); rdir = f"{RUN_DIR}/{rid}"; meta_p = f"{rdir}/meta.json"
    _m0 = read_json(meta_p, {})
    if not FORCE_RERUN and _m0.get("status") == "ok" and _m0.get("metric_version") == METRIC_VERSION:   # run đo bằng phiên bản cũ -> chạy lại
        tqdm.write(f"[skip] {rid}"); return "skipped"
    os.makedirs(rdir, exist_ok=True)
    t0 = time.time()
    tqdm.write("=" * 78); tqdm.write(f"[run] {rid}")
    try:
        if os.path.exists(f"{rdir}/error.txt"):
            os.remove(f"{rdir}/error.txt")
        write_json(meta_p, dict(cfg=cfg, status="running"))
        W = get_world(world_cfg(cfg["L"], cfg["level"], cfg["overlap"], cfg["len_scale"], cfg["seed"],
                                cfg["len_scale_X"], cfg["share_X"]))
        fam, task, d = cfg["family"], cfg["task"], cfg["d"]
        torch.manual_seed(1000 + cfg["seed"])
        model = build_model(fam, d, task, W)
        n_par = model.n_params_nonemb()
        tqdm.write(f"  tham số không-embedding: {n_par / 1e6:.2f}M | fertility: {[round(x, 2) for x in W['meta']['fertility']]}")
        measure(model, W, rdir, "init", fam)                                   # null: mô hình chưa huấn luyện
        info = train_model(model, fam, task, W, STEPS, cfg["seed"])
        val = eval_lm_loss(model, W)
        measure(model, W, rdir, "trained", fam, masks=(fam == "diffusion"))
        extra = {}
        if cfg["lens"]:
            assert cfg["overlap"] == 0 and fam in ("decoder", "moe")
            save_npz(rdir, "lens", dict(M=logit_lens_mass(model, W["eval_ids"], W["tok_lang"])))
        del model
        if cfg["twins"]:
            extra["twin_seconds"] = run_twins(cfg, W, rdir)
        write_json(meta_p, dict(cfg=cfg, status="ok", seconds=round(time.time() - t0, 1), params_nonemb=n_par,
                                final_lm_loss=info["final_lm_loss"], eval_lm_loss=val, steps=info["steps"], batch=BATCH, lr_scale=LR_SCALE,
                                fertility=W["meta"]["fertility"], world=W["meta"]["id"], verify=W["meta"]["verify"],
                                finished=datetime.now().isoformat(timespec="seconds"),
                                metric_version=METRIC_VERSION, versions=dict(torch=torch.__version__), **extra))
        tqdm.write(f"[done] {rid} trong {(time.time() - t0) / 60:.1f} phút  (loss LM ≈ {info['final_lm_loss']:.3f}, eval {val})")
        return "ok"
    except Exception as e:
        with open(f"{rdir}/error.txt", "w") as f:
            f.write(traceback.format_exc())
        write_json(meta_p, dict(cfg=cfg, status="error"))
        tqdm.write(f"[ERROR] {rid}: {repr(e)[:300]}  (chi tiết: error.txt) -> chuyển sang run kế tiếp")
        return "error"
    finally:
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

def run_grid(name, limit=None):
    cfgs = grid_configs(name)[: limit]
    rep = {}
    bar = tqdm(cfgs, desc=f"lưới {name}", unit="run", dynamic_ncols=True)
    for c in bar:
        rid = run_id(c)
        rep[rid] = run_one(c)
        bar.set_postfix({s: sum(v == s for v in rep.values()) for s in ("ok", "skipped", "error")}, refresh=True)
    print("\nTổng kết", name, {s: sum(v == s for v in rep.values()) for s in ("ok", "skipped", "error")})
    bad = [k for k, v in rep.items() if v == "error"]
    if bad:
        print("Lỗi ở:", bad, "-> sửa rồi chạy lại cell này; run đã xong sẽ được bỏ qua.")
    return rep

def show_status():
    import pandas as pd
    rows = []
    for g in ALL_GRIDS:
        cfgs = grid_configs(g)
        st = [read_json(f"{RUN_DIR}/{run_id(c)}/meta.json", {}).get("status", "missing") for c in cfgs]
        secs = [read_json(f"{RUN_DIR}/{run_id(c)}/meta.json", {}).get("seconds") for c in cfgs]
        secs = [s for s in secs if s]
        rows.append(dict(grid=g, runs=len(cfgs), ok=st.count("ok"), error=st.count("error"), running=st.count("running"),
                         mean_min=round(float(np.mean(secs)) / 60, 1) if secs else None))
    return pd.DataFrame(rows)

In [22]:
# ============================================================
# Cell C3 — Trạng thái hiện tại (đọc từ STUDY_DIR, không cần GPU)
# ============================================================
show_status()

,grid,runs,ok,error,running,mean_min
0,gate_g2,3,0,0,0,None
1,smoke,5,0,0,0,None
2,p1p2,108,0,0,0,None
3,balanced,24,0,0,0,None
4,moe_diff,36,0,0,0,None
5,p3s2,180,0,0,0,None
6,s4,24,0,0,0,None
7,s5,30,0,0,0,None
8,heldout_p4,48,0,0,0,None


In [25]:
# ============================================================
# Cell C3b — Chẩn đoán nhanh (~1 phút, không train): U_0/log L ở tầng embedding, cách đo CŨ vs MỚI
# Mô hình CHƯA huấn luyện, thế giới của gate seed 0. Kỳ vọng: cũ ≈ 0, mới ≈ 1 (overlap 0% -> chữ cái lộ ngôn ngữ).
# ============================================================
_c = make_cfg("diag", seed=0)
_W = get_world(world_cfg(_c["L"], _c["level"], _c["overlap"], _c["len_scale"], _c["seed"], _c["len_scale_X"], _c["share_X"]))
torch.manual_seed(1000)
_m = build_model("decoder", D_MAIN, "SEM", _W)
_Z, _ = extract_eval(_m, _W["eval_ids"], _W["eval_wid"], desc="[diag] trích eval")
_Z0 = _Z[0].to(DEVICE, torch.float32)                      # [L, N, d] tầng 0 = embedding
_L = _Z0.shape[0]
_old = _standardize(_Z0);           _old = _old - _old.mean(0, keepdim=True)
_new = _standardize_keep_lang(_Z0); _new = _new - _new.mean(0, keepdim=True)
for _name, _Zt in (("CŨ  (trừ tâm từng ngôn ngữ)", _old), ("MỚI (giữ tâm từng ngôn ngữ)", _new)):
    _u, _acc = usable_info_lin(_Zt, SEED)
    print(f"{_name}: U_0/log L = {_u / math.log(_L):+.3f} | accuracy đoán ngôn ngữ = {_acc:.3f} (đoán mò = {1 / _L:.2f})")
del _m, _Z, _Z0, _old, _new
if DEVICE == "cuda":
    torch.cuda.empty_cache()


sinh pool (render):   0%|                                                           | 0/400000 [00:00<?, ?it/s…

BPE merges:   0%|                                                                     | 0/7947 [00:00<?, ?it/s…

BPE encode từ:   0%|                                                                 | 0/95191 [00:00<?, ?it/s…

ghép token cho pool:   0%|                                                          | 0/400000 [00:00<?, ?it/s…

kiểm (G) bằng parser:   0%|                                                            | 0/200 [00:00<?, ?it/s…

[world] L4_M4_ov0_ls1_sheq_s0: pool=400000  vocab=9106  fertility=[1.36, 1.38, 1.35, 1.41]  roundtrip_fail=0.0000  parsed_by_other=0.0000  (103.8s)


[diag] trích eval:   0%|                                                             | 0/12 [00:00<?, ?batch/s…

CŨ  (trừ tâm từng ngôn ngữ): U_0/log L = -0.050 | accuracy đoán ngôn ngữ = 0.241 (đoán mò = 0.25)
MỚI (giữ tâm từng ngôn ngữ): U_0/log L = +0.643 | accuracy đoán ngôn ngữ = 0.831 (đoán mò = 0.25)


In [26]:
# ============================================================
# Cell C4 — CỔNG G2: một cấu hình decoder M4/SEM, 3 seed.  Phải có U_0/log L ≈ 1 ở embedding và có eo.
# Sau khi chạy: đo thời gian thực mỗi run và nhân lại ngân sách cho toàn bộ lưới (in ở cuối).
# (với SMOKE=True đặt RUN_GRIDS = ["smoke"])
# ============================================================
for _g in RUN_GRIDS:
    run_grid(_g)

lưới gate_g2:   0%|                                                                     | 0/3 [00:00<?, ?run/s…

[run] gate_g2__decoder_M4_SEM_d192_L4_ov0_ls1_sheq_s0
  tham số không-embedding: 3.54M | fertility: [1.36, 1.38, 1.35, 1.41]


[init] trích eval:   0%|                                                             | 0/12 [00:00<?, ?batch/s…

[init] số đo:   0%|                                                                    | 0/9 [00:00<?, ?tầng/s…

[init] trích probe S3:   0%|                                                          | 0/8 [00:00<?, ?batch/s…

...

train decoder/SEM:   0%|                                                             | 0/750 [00:00<?, ?step/s…

[trained] trích eval:   0%|                                                          | 0/12 [00:00<?, ?batch/s…

[trained] số đo:   0%|                                                                 | 0/9 [00:00<?, ?tầng/s…

[trained] trích probe S3:   0%|                                                       | 0/8 [00:00<?, ?batch/s…

[done] gate_g2__decoder_M4_SEM_d192_L4_ov0_ls1_sheq_s2 trong 6.4 phút  (loss LM ≈ 5.393, eval 5.065998186776116)

Tổng kết gate_g2 {'ok': 3, 'skipped': 0, 'error': 0}


In [27]:
# ============================================================
# Cell C5 — Báo cáo cổng G2 + dự toán thời gian toàn lưới
# ============================================================
def gate_g2_report(grid="gate_g2"):
    rows = []
    for c in grid_configs(grid):
        rdir = f"{RUN_DIR}/{run_id(c)}"
        m = read_json(f"{rdir}/meta.json", {})
        tr, ini = load_npz(rdir, "trained"), load_npz(rdir, "init")
        if m.get("status") != "ok" or tr is None:
            continue
        if m.get("metric_version") != METRIC_VERSION:
            print(f"[bỏ qua] {run_id(c)}: kết quả đo bằng phiên bản cũ -> chạy lại cell C4"); continue
        L = c["L"]
        Gm = np.nanmean(tr["G"], 0); Gi = np.nanmean(ini["G"], 0)
        k = int(np.nanargmax(Gm))
        rows.append(dict(seed=c["seed"], U0_over_logL=float(tr["U_lin"][0] / math.log(L)), U0_mlp_over_logL=float(tr["U_mlp"][0] / math.log(L)), acc0=float(tr["U_acc"][0]), k_star=k, K=len(Gm) - 1,
                         G_peak=float(Gm[k]), edge_depth=float(Gm[k] - max(Gm[0], Gm[-1])),
                         edge_depth_init=float(Gi.max() - max(Gi[0], Gi[-1])),
                         interior=bool(0 < k < len(Gm) - 1), loss=m["final_lm_loss"], eval_loss=m["eval_lm_loss"], minutes=m["seconds"] / 60))
    import pandas as pd
    df = pd.DataFrame(rows)
    if df.empty:
        print("Chưa có kết quả gate."); return df
    print(df.round(3).to_string(index=False))
    ok_u = bool((df.U0_over_logL > 0.9).all()); ok_w = bool((df.interior & (df.edge_depth > 0.05) & (df.edge_depth > df.edge_depth_init + 0.03)).all())
    print(f"\nG2: U_0/log L ≈ 1 -> {'OK' if ok_u else 'KHÔNG ĐẠT'} | có eo (đỉnh G nằm trong thân, sâu > 0.05, hơn init) -> {'OK' if ok_w else 'KHÔNG ĐẠT'}")
    mean_min = float(df.minutes.mean())
    tot = sum(len(grid_configs(g)) for g in ALL_GRIDS if g not in ("gate_g2", "smoke"))
    print(f"Thời gian trung bình/run ≈ {mean_min:.1f} phút -> ước tính toàn lưới ({tot} run, chưa tính sinh đôi) ≈ {mean_min * tot / 60:.0f} giờ GPU")
    if not (ok_u and ok_w):
        print("-> Nếu không có eo: sửa bộ sinh (độ khác biệt, kích thước pool, số bước) trước khi chạy lưới.")
    return df

gate_g2_report("smoke" if SMOKE else "gate_g2")

 seed  U0_over_logL  U0_mlp_over_logL  acc0  k_star  K  G_peak  edge_depth  edge_depth_init  interior  loss  eval_loss  minutes
    0           1.0               1.0   1.0       1  8   0.429       0.151            0.000      True 5.316      5.026    4.283
    1           1.0               1.0   1.0       1  8   0.422       0.138            0.000      True 5.379      5.067    6.957
    2           1.0               1.0   1.0       1  8   0.586       0.217            0.002      True 5.393      5.066    6.363

G2: U_0/log L ≈ 1 -> OK | có eo (đỉnh G nằm trong thân, sâu > 0.05, hơn init) -> OK
Thời gian trung bình/run ≈ 5.9 phút -> ước tính toàn lưới (450 run, chưa tính sinh đôi) ≈ 44 giờ GPU


,seed,U0_over_logL,U0_mlp_over_logL,acc0,k_star,K,G_peak,edge_depth,edge_depth_init,interior,loss,eval_loss,minutes
0,0,0.999980,0.999997,1.0,1,8,0.428764,0.151411,0.000000,True,5.315920,5.026338,4.283333
1,1,0.999981,0.999997,1.0,1,8,0.421679,0.137594,0.000000,True,5.379071,5.067206,6.956667
2,2,0.999978,0.999997,1.0,1,8,0.585677,0.217492,0.002343,True,5.392773,5.065998,6.363333


In [ ]:
# ============================================================
# Cell C6 — CHẠY các lưới còn lại (mỗi lưới một lần gọi; có resume nên chạy lại bao nhiêu lần cũng được)
# Bỏ comment từng dòng khi cần. Thứ tự gợi ý theo giá trị khoa học / chi phí.
# ============================================================
run_grid("p1p2")        # 108 run: P1 (decoder, encoder), P2
run_grid("balanced")    #  24 run: cánh cân bằng giao diện (A1)
run_grid("moe_diff")    #  36 run: MoE + diffusion (P1, S6)
run_grid("p3s2")        # 180 run (+72 sinh đôi): P3, S2, S3
run_grid("s4")          #  24 run: S4 (pivot)
run_grid("s5")          #  30 run: S5
run_grid("heldout_p4")  #  48 run: giữ ra cho P4 (đầu vào của Study 5)
show_status()

lưới p1p2:   0%|                                                                      | 0/108 [00:00<?, ?run/s…

[run] p1p2__decoder_M0_LM_d192_L4_ov0_ls1_sheq_s0
  tham số không-embedding: 3.54M | fertility: [1.3, 1.3, 1.3, 1.31]


[init] trích eval:   0%|                                                              | 0/4 [00:00<?, ?batch/s…

[init] số đo:   0%|                                                                    | 0/9 [00:00<?, ?tầng/s…

[init] trích probe S3:   0%|                                                          | 0/4 [00:00<?, ?batch/s…

...

[init] trích eval:   0%|                                                              | 0/4 [00:00<?, ?batch/s…

[init] số đo:   0%|                                                                    | 0/9 [00:00<?, ?tầng/s…

[init] trích probe S3:   0%|                                                          | 0/4 [00:00<?, ?batch/s…

train decoder/LM:   0%|                                                              | 0/750 [00:00<?, ?step/s…

KeyboardInterrupt: 

## Phần D — Phân tích, kiểm định giả thuyết, hình, báo cáo (không cần GPU)

Chỉ đọc `STUDY_DIR`; ở kernel mới chỉ cần chạy **Cell A1** rồi các cell của Phần D. Quy ước giữ nguyên Study 1:
- **Waist** (mỗi run): đỉnh của $\bar G_k$ nằm trong thân; cận dưới CI95 (bootstrap theo nội dung) của *edge depth* $S_{edge}=\max_k \bar G_k-\max(\bar G_0,\bar G_K)$ vượt 0,05; đỉnh CKA rơi vào dải waist của G (nới ±1 tầng); và vượt mô hình chưa huấn luyện (`init`, cùng kiến trúc/seed).
- $\rho^*=k^*/K$, độ rộng $w$ = tỉ lệ tầng có $\bar G_k\ge \bar G_{k^*}-0{,}25\,S$.
- **Đơn vị thống kê là seed** (mỗi seed = một thế giới ngôn ngữ độc lập, ghép cặp qua điều kiện): Wilcoxon chính xác ghép cặp, TOST cho dự đoán "không đổi" (biên $\Delta\rho^*=0{,}10$; $\Delta A=\max(0{,}05,\ 1\,SD_{seed})$), Jonckheere–Terpstra cho thứ tự, hiệu chỉnh Holm trong từng giả thuyết.
- Dải mờ trong hình = CI95 (t) qua seed.

In [ ]:
# ============================================================
# Cell D1 — Tải kết quả + đặc trưng waist cho từng run
# ============================================================
import pandas as pd
from scipy import stats
warnings.filterwarnings("ignore", category=RuntimeWarning)

FAMILIES = ["decoder", "encoder", "moe", "diffusion"]
TASKS = ["LM", "SEM", "SURF"]
LEVELS = ("M0", "M2", "M4")
WAIST_S_FLOOR = 0.05      # cận dưới CI95 của edge depth phải vượt ngưỡng này
CKA_SLACK = 1             # đỉnh CKA được phép nằm ngoài dải waist của G tối đa 1 tầng
RHO_MARGIN = 0.10         # SESOI cho vị trí waist

def _rj(p):
    if not os.path.exists(p):
        return None
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def _ld(rid, name):
    p = f"{RUN_DIR}/{rid}/{name}.npz"
    if not os.path.exists(p):
        return None
    with np.load(p, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def scan_runs():
    rows = []
    for rid in sorted(os.listdir(RUN_DIR)):
        m = _rj(f"{RUN_DIR}/{rid}/meta.json")
        if m and m.get("status") == "ok":
            c = m["cfg"]
            rows.append(dict(rid=rid, **c, seconds=m["seconds"], loss=m["final_lm_loss"], params=m["params_nonemb"],
                             fert_mean=float(np.mean(m["fertility"])), fert_X=float(m["fertility"][0]),
                             fert_list=m["fertility"], has_twins=os.path.exists(f"{RUN_DIR}/{rid}/twins.npz")))
    return pd.DataFrame(rows)

_CV = {}
def cv(rid, variant="trained"):
    """Đường cong (trung bình qua cặp ngôn ngữ) của một run/variant, có cache."""
    key = (rid, variant)
    if key not in _CV:
        d = _ld(rid, variant)
        if d is None:
            _CV[key] = None
        else:
            o = dict(G=np.nanmean(d["G"], 0), D=np.nanmean(d["D"], 0), CKA=np.nanmean(d["CKA"], 0), A=np.nanmean(d["A"], 0),
                     gc=d["g_content"].astype(np.float32), U=d["U_lin"], Um=d["U_mlp"], Apair=d["A"], Gpair=d["G"])
            if "G_rank" in d:
                o["Grank"], o["rank_counts"] = d["G_rank"], d["rank_counts"]
            _CV[key] = o
    return _CV[key]

def boot_G(gc, n_boot=None, seed=SEED, chunk=100):
    """gc [K1,N] (g theo nội dung, đã trung bình qua cặp) -> các đường G bootstrap theo nội dung [B,K1]."""
    n_boot = n_boot or N_BOOT
    K1, N = gc.shape
    rng = np.random.default_rng(seed)
    out = np.empty((n_boot, K1), np.float32)
    for s in range(0, n_boot, chunk):
        m = min(chunk, n_boot - s)
        out[s:s + m] = gc[:, rng.integers(0, N, size=(m, N))].mean(-1).T
    return out

def curve_features(Gm):
    Gm = np.nan_to_num(np.asarray(Gm, np.float64), nan=0.0)
    K1 = Gm.shape[-1]
    ks = Gm.argmax(-1)
    Gk = np.take_along_axis(Gm, ks[..., None], -1)[..., 0]
    S = Gk - 0.5 * (Gm[..., 0] + Gm[..., -1])
    w = (Gm >= (Gk - 0.25 * S)[..., None]).mean(-1)
    return ks, ks / (K1 - 1), S, w

def edge_depth(Gm):
    Gm = np.nan_to_num(np.asarray(Gm, np.float64), nan=0.0)
    return Gm.max(-1) - np.maximum(Gm[..., 0], Gm[..., -1])

def _slope(y, x):
    return float(np.polyfit(x, y, 1)[0]) if len(x) >= 2 else np.nan

def waist_from_curves(Gm, Gb, cka):
    Gm = np.nan_to_num(np.asarray(Gm, np.float64), nan=0.0); K1 = len(Gm)
    k, rho, S, w = (x[0] for x in curve_features(Gm[None])); k = int(k)
    kb, rb, Sb, wb = curve_features(Gb)
    ci = lambda x: tuple(float(v) for v in np.percentile(x, [2.5, 97.5]))
    Se, Se_ci = float(edge_depth(Gm)), ci(edge_depth(Gb))
    band = Gm >= (Gm[k] - 0.25 * S)
    k_cka = int(np.nanargmax(cka)) if np.isfinite(cka).any() else -1
    cka_in = bool(k_cka >= 0 and band[max(0, k_cka - CKA_SLACK): k_cka + CKA_SLACK + 1].any())
    interior = bool(0 < k < K1 - 1)
    x = np.arange(K1)
    return dict(K=K1 - 1, k_star=k, rho_star=float(rho), rho_lo=ci(rb)[0], rho_hi=ci(rb)[1], S=float(S), w=float(w),
                S_edge=Se, S_edge_lo=Se_ci[0], S_edge_hi=Se_ci[1], interior=interior,
                G_at_kstar=float(Gm[k]), G_first=float(Gm[0]), G_last=float(Gm[-1]),
                k_star_cka=k_cka, cka_in_band=cka_in, has_waist_curve=bool(interior and Se_ci[0] > WAIST_S_FLOOR and cka_in),
                slope_left=_slope(Gm[:k + 1], x[:k + 1]), slope_right=_slope(Gm[k:], x[k:]))

_FEAT = {}
def feat(rid, variant="trained"):
    key = (rid, variant)
    if key not in _FEAT:
        c = cv(rid, variant)
        _FEAT[key] = None if c is None else waist_from_curves(c["G"], boot_G(c["gc"]), c["CKA"])
    return _FEAT[key]

def build_feature_table(save=True):
    runs = scan_runs()
    if runs.empty:
        return runs
    rows = []
    for r in runs.itertuples():
        f, fi = feat(r.rid, "trained"), feat(r.rid, "init")
        if f is None:
            continue
        c = cv(r.rid, "trained")
        row = {**r._asdict(), **f}
        row["S_edge_init_hi"] = fi["S_edge_hi"] if fi else np.nan
        row["above_init"] = bool(f["S_edge_lo"] > fi["S_edge_hi"]) if fi else True
        row["has_waist"] = bool(f["has_waist_curve"] and row["above_init"])
        row["U0_over_logL"] = float(c["U"][0] / math.log(r.L))
        row["U_at_kstar_over_logL"] = float(c["U"][f["k_star"]] / math.log(r.L))
        row["Umlp_at_kstar_over_logL"] = float(c["Um"][f["k_star"]] / math.log(r.L))
        row["ln_params"] = math.log(r.params)
        if r.family == "diffusion":
            for t in MASK_LEVELS[1:]:
                ft = feat(r.rid, f"t{t:g}")
                if ft:
                    row[f"rho_t{t:g}"], row[f"S_edge_t{t:g}"], row[f"G_last_t{t:g}"], row[f"kstar_t{t:g}"] = ft["rho_star"], ft["S_edge"], ft["G_last"], ft["k_star"]
        rows.append(row)
    df = pd.DataFrame(rows).drop(columns=["Index"], errors="ignore")
    if save and len(df):
        df.drop(columns=["fert_list"]).to_csv(f"{ANALYSIS_DIR}/features_all_runs.csv", index=False)
    return df

In [ ]:
# ============================================================
# Cell D2 — Tiện ích thống kê (Wilcoxon chính xác, TOST, JT, bước-so-với-tuyến-tính, Holm)
# ============================================================
def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); out = np.empty_like(ps); run = 0.0
    for i, j in enumerate(o):
        run = max(run, (len(ps) - i) * ps[j]); out[j] = min(1.0, run)
    return out

def wilcoxon_p(x, y=None, alt="greater"):
    x = np.asarray(x, float) if y is None else np.asarray(x, float) - np.asarray(y, float)
    x = x[np.isfinite(x)]
    if len(x) < 4 or np.all(x == 0):
        return np.nan
    try:
        return float(stats.wilcoxon(x, alternative=alt).pvalue)         # chính xác khi n nhỏ và không có số 0
    except Exception:
        return np.nan

def tost_p(x, margin):
    """TOST cho 'trung bình ≈ 0 trong ±margin' (p nhỏ = bằng nhau theo nghĩa thực dụng)."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]; n = len(x)
    if n < 3:
        return np.nan
    m, se = x.mean(), x.std(ddof=1) / math.sqrt(n)
    if se == 0:
        return 0.0 if abs(m) < margin else 1.0
    return float(max(1 - stats.t.cdf((m + margin) / se, n - 1), stats.t.cdf((m - margin) / se, n - 1)))

def jonckheere_terpstra(groups):
    """groups xếp theo thứ tự tăng giả thuyết (nhóm sau lớn hơn). Trả về (J, z, p một phía 'tăng')."""
    g = [np.asarray(x, float)[np.isfinite(x)] for x in groups]
    J = 0.0
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            d = g[j][None, :] - g[i][:, None]
            J += float((d > 0).sum() + 0.5 * (d == 0).sum())
    n = np.array([len(x) for x in g]); N = n.sum()
    mu = (N ** 2 - (n ** 2).sum()) / 4.0
    var = (N ** 2 * (2 * N + 3) - (n ** 2 * (2 * n + 3)).sum()) / 72.0
    z = (J - mu) / math.sqrt(var) if var > 0 else np.nan
    return J, float(z), float(1 - stats.norm.cdf(z))

def _aicc(sse, n, k):
    return n * math.log(max(sse, 1e-12) / n) + 2 * k + 2 * k * (k + 1) / max(n - k - 1, 1)

def step_vs_linear(x, y):
    """x (vd. log2 d), y: quan sát cấp seed. So mô hình bậc thang một điểm gãy với đường thẳng bằng AICc.
    Trả về (ΔAICc = AICc_tuyến_tính - AICc_bậc_thang, chỉ số điểm gãy trong các x duy nhất)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(y); x, y = x[ok], y[ok]; n = len(y)
    if n < 8 or len(np.unique(x)) < 3:
        return np.nan, -1
    b, a = np.polyfit(x, y, 1)
    aic_lin = _aicc(float(np.square(y - (a + b * x)).sum()), n, 3)
    ux = np.unique(x); best = (np.inf, -1)
    for i in range(len(ux) - 1):
        lo = x <= ux[i]
        sse = float(np.square(y[lo] - y[lo].mean()).sum() + np.square(y[~lo] - y[~lo].mean()).sum())
        if sse < best[0]:
            best = (sse, i)
    return aic_lin - _aicc(best[0], n, 4), best[1]

def tcrit_band(M):
    """M [R,K] -> (mean, lo, hi) CI95 (t) qua R run (seed)."""
    M = np.asarray(M, float); R = M.shape[0]
    m = np.nanmean(M, 0)
    if R < 2:
        return m, m, m
    h = stats.t.ppf(0.975, R - 1) * np.nanstd(M, 0, ddof=1) / math.sqrt(R)
    return m, m - h, m + h

def sel(df, **kw):
    out = df
    for k, v in kw.items():
        out = out[out[k].isin(v)] if isinstance(v, (list, tuple, set)) else out[out[k] == v]
    return out

def _has(x):
    """pandas đổi None thành NaN: NaN không phải 'có giá trị' (vd. share_X, len_scale_X của lưới không dùng)."""
    return x is not None and not (isinstance(x, float) and math.isnan(x))

def _res(hyp, verdict, evidence, table=None):
    return dict(hyp=hyp, verdict=verdict, evidence=evidence, table=table)

NO_DATA = "chưa có dữ liệu"

In [ ]:
# ============================================================
# Cell D3 — Kiểm định giả thuyết P1, P2, P3, S2, S3, S4, S5, S6
# Mỗi hàm trả về dict(hyp, verdict, evidence, table). verdict ∈ {ủng hộ, không ủng hộ, chưa kết luận, chưa có dữ liệu}.
# ============================================================
def test_P1(df):
    """P1: cả bốn họ có waist (G_k, D_k dip so với mô hình chưa huấn luyện); waist còn khi giao diện cân bằng (loại A1)."""
    rows = []
    d0 = sel(df, overlap=0, level="M4", task=["LM", "SEM"])
    for fam in FAMILIES:
        s = sel(d0, family=fam)
        if s.empty:
            continue
        diff = (s.S_edge - s.S_edge_init_hi).values
        p = wilcoxon_p(diff)
        rows.append(dict(part="4 họ (overlap 0%, M4, LM+SEM)", group=fam, n=len(s), n_waist=int(s.has_waist.sum()), frac=float(s.has_waist.mean()),
                         median_S_edge=float(s.S_edge.median()), median_S_edge_init_hi=float(s.S_edge_init_hi.median()), p_vs_init=p,
                         frac_Ushape=float(((s.slope_left > 0) & (s.slope_right < 0)).mean()), median_rho=float(s.rho_star.median())))
    for lv in ("M0", "M2", "M4"):
        s = sel(df, family="decoder", overlap=0, level=lv, task=["LM", "SEM"], grid="p1p2")
        b = sel(df, family="decoder", overlap=100, level=lv, task=["LM", "SEM"], grid="balanced")
        for name, x in (("overlap 0%", s), ("overlap 100% (giao diện cân bằng)", b)):
            if len(x):
                rows.append(dict(part=f"A1 decoder {lv}", group=name, n=len(x), n_waist=int(x.has_waist.sum()), frac=float(x.has_waist.mean()),
                                 median_S_edge=float(x.S_edge.median()), median_S_edge_init_hi=float(x.S_edge_init_hi.median()),
                                 p_vs_init=wilcoxon_p((x.S_edge - x.S_edge_init_hi).values), frac_Ushape=float(((x.slope_left > 0) & (x.slope_right < 0)).mean()),
                                 median_rho=float(x.rho_star.median())))
    t = pd.DataFrame(rows)
    if t.empty:
        return _res("P1", NO_DATA, "")
    fam = t[t.part.str.startswith("4 họ")]
    ok_fam = {r.group: bool(r.frac >= 0.8 and r.p_vs_init < 0.05) for r in fam.itertuples()}
    bal = t[(t.part == "A1 decoder M4") & (t.group.str.startswith("overlap 100"))]
    ok_bal = bool(len(bal) and bal.iloc[0].frac >= 0.8 and bal.iloc[0].p_vs_init < 0.05)
    miss = [f for f in FAMILIES if f not in ok_fam]
    ev = f"họ có waist: {ok_fam}; còn waist khi overlap 100% (M4): {ok_bal if len(bal) else NO_DATA}" + (f"; thiếu dữ liệu: {miss}" if miss else "")
    if miss or not len(bal):
        return _res("P1", "chưa kết luận", ev, t)
    return _res("P1", "ủng hộ" if (all(ok_fam.values()) and ok_bal) else "không ủng hộ", ev + ("" if ok_bal else " -> A1 (giao diện) chưa bị loại"), t)

def test_P2(df):
    """P2: vị trí/độ rộng/độ sâu waist đi theo tác vụ: rộng nhất ở SEM, hẹp hoặc dịch ở SURF; kỳ vọng bị bác nếu eo cố định bất kể tác vụ."""
    rows = []
    for fam in ("decoder", "encoder"):
        for lv in LEVELS:
            s = sel(df, family=fam, level=lv, overlap=0, grid="p1p2")
            if s.empty:
                continue
            piv = {m: s.pivot_table(index="seed", columns="task", values=m) for m in ("w", "S_edge", "rho_star")}
            if not all(t in piv["w"].columns for t in TASKS):
                continue
            g = [piv["w"][t].dropna().values for t in ("SURF", "LM", "SEM")]
            J, z, pj = jonckheere_terpstra(g)
            dS = (piv["S_edge"]["SEM"] - piv["S_edge"]["SURF"]).values
            dw = (piv["w"]["SEM"] - piv["w"]["SURF"]).values
            dr = (piv["rho_star"]["SEM"] - piv["rho_star"]["SURF"]).values
            rows.append(dict(family=fam, level=lv, n_seed=len(dS), JT_p_width=pj, p_depth_SEM_gt_SURF=wilcoxon_p(dS), p_width_SEM_gt_SURF=wilcoxon_p(dw),
                             p_rho_shift=wilcoxon_p(dr, alt="two-sided"), mean_drho=float(np.nanmean(dr)),
                             tost_rho_fixed=tost_p(dr, RHO_MARGIN), tost_depth_equal=tost_p(dS, 0.05),
                             mean_w={t: round(float(piv["w"][t].mean()), 3) for t in TASKS}))
    t = pd.DataFrame(rows)
    if t.empty:
        return _res("P2", NO_DATA, "")
    # Holm TRONG từng mức (M4 là mức chính: 2 họ -> với 6 seed p_min = 0.0156 x 2 = 0.031 < 0.05 vẫn đạt được);
    # M0/M2 được báo cáo để so sánh (thăm dò).
    for c in ("JT_p_width", "p_depth_SEM_gt_SURF", "p_width_SEM_gt_SURF", "p_rho_shift"):
        t[c + "_holm"] = np.nan
        for lv in t.level.unique():
            ix = t.level == lv
            t.loc[ix, c + "_holm"] = holm(t.loc[ix, c].fillna(1.0).values)
    t["task_matters"] = (t.JT_p_width_holm < 0.05) | (t.p_depth_SEM_gt_SURF_holm < 0.05) | ((t.p_rho_shift_holm < 0.05) & (t.mean_drho.abs() > RHO_MARGIN))
    t["waist_fixed"] = (t.tost_rho_fixed < 0.05) & (t.tost_depth_equal < 0.05)
    m4 = t[t.level == "M4"]
    ev = f"M4: nhạy với tác vụ {m4.set_index('family').task_matters.to_dict()}; eo cố định (TOST) {m4.set_index('family').waist_fixed.to_dict()}"
    if m4.empty:
        return _res("P2", "chưa kết luận", ev, t)
    if m4.waist_fixed.all():
        return _res("P2", "không ủng hộ", ev, t)
    return _res("P2", "ủng hộ" if m4.task_matters.all() else "chưa kết luận", ev, t)

def _delta_A(r):
    """ΔA[pair, tầng] = A_joint - A_indep cho một run có sinh đôi."""
    c, tw = cv(r.rid, "trained"), _ld(r.rid, "twins")
    return None if (c is None or tw is None) else c["Apair"] - tw["A_indep"]

def test_P3(df):
    """P3: huấn luyện chung thêm căn chỉnh so với sinh đôi đơn ngữ: ΔA > 0, đỉnh ở waist, giảm khi capacity tăng, đúng cho mọi cặp."""
    s = df[df.has_twins & df.grid.isin(["p3s2", "smoke"])]
    if s.empty:
        return _res("P3", NO_DATA, "")
    rows, dA_curves = [], {}
    for r in s.itertuples():
        dA = _delta_A(r)
        if dA is None:
            continue
        m = np.nanmean(dA, 0); k = r.k_star
        rows.append(dict(rid=r.rid, d=r.d, L=r.L, seed=r.seed, ln_params=r.ln_params, dA_kstar=float(m[k]), dA_peak_layer=int(np.nanargmax(m)), k_star=k,
                         frac_pairs_pos=float((dA[:, k] > 0).mean()), dA_max=float(np.nanmax(m)), A_joint_k=float(np.nanmean(cv(r.rid)["Apair"][:, k])),
                         A_indep_k=float(np.nanmean(_ld(r.rid, "twins")["A_indep"][:, k]))))
    t = pd.DataFrame(rows)
    if t.empty:
        return _res("P3", NO_DATA, "")
    sd_seed = float(t.groupby(["d", "L"]).dA_kstar.std().mean())
    margin = max(0.05, sd_seed if np.isfinite(sd_seed) else 0.05)
    p_pos = wilcoxon_p(t.dA_kstar.values)
    p_tost = tost_p(t.dA_kstar.values, margin)
    sl = stats.linregress(t.ln_params, t.dA_kstar) if len(t) >= 5 else None
    p_slope = float(sl.pvalue / 2 if sl.slope < 0 else 1 - sl.pvalue / 2) if sl else np.nan
    at_waist = float((np.abs(t.dA_peak_layer - t.k_star) <= 1).mean())
    frac_pairs = float(t.frac_pairs_pos.mean())
    ev = (f"ΔA tại waist: median {t.dA_kstar.median():.3f} (Wilcoxon p={p_pos:.3g}); TOST≈0 (biên {margin:.3f}) p={p_tost:.3g}; "
          f"độ dốc theo log B = {sl.slope:.3f} (p một phía={p_slope:.3g}); đỉnh ΔA trong ±1 tầng quanh waist: {at_waist:.0%}; cặp có ΔA>0: {frac_pairs:.0%}") if sl else "quá ít run có sinh đôi"
    if not sl:
        return _res("P3", "chưa kết luận", ev, t)
    if p_tost < 0.05:
        return _res("P3", "không ủng hộ", ev + " -> ΔA≈0: A2 (hội tụ Platonic)", t)
    ok = (p_pos < 0.05) and (p_slope < 0.05) and at_waist >= 0.5 and frac_pairs >= 0.9
    return _res("P3", "ủng hộ" if ok else "chưa kết luận", ev, t)

def test_S2(df):
    """S2: căn chỉnh tại waist là hàm bậc thang theo capacity (và số ngôn ngữ), phương sai giữa seed đạt đỉnh ở bước nhảy."""
    s = sel(df, grid="p3s2", family="decoder")
    if s.empty:
        return _res("S2", NO_DATA, "")
    rows = []
    for L in sorted(s.L.unique()):
        x = s[s.L == L]
        if x.d.nunique() < 4:
            continue
        dAIC, i = step_vs_linear(np.log2(x.d.values), x.G_at_kstar.values)
        ds = sorted(x.d.unique())
        grp = [x[x.d == d].G_at_kstar.values for d in ds]
        bf = stats.levene(*grp, center="median").pvalue if all(len(g) > 1 for g in grp) else np.nan
        sds = [np.nanstd(g, ddof=1) for g in grp]
        peak = int(np.nanargmax(sds)) if np.isfinite(sds).any() else -1
        rows.append(dict(L=L, n=len(x), dAICc_step_vs_linear=dAIC, step_after_d=ds[i] if i >= 0 else None, levene_p=bf,
                         var_peak_d=ds[peak] if peak >= 0 else None, var_peak_adjacent_to_step=bool(i >= 0 and peak in (i, i + 1))))
    t = pd.DataFrame(rows)
    if t.empty:
        return _res("S2", NO_DATA, "cần >= 4 giá trị d")
    t["staircase"] = (t.dAICc_step_vs_linear > 2) & t.var_peak_adjacent_to_step
    ev = f"L có dạng bậc thang + đỉnh phương sai ở bước nhảy: {int(t.staircase.sum())}/{len(t)}; ΔAICc: {t.set_index('L').dAICc_step_vs_linear.round(2).to_dict()}"
    return _res("S2", "ủng hộ" if t.staircase.sum() >= 2 else ("không ủng hộ" if (t.dAICc_step_vs_linear < -2).all() else "chưa kết luận"), ev, t)

def _inverted_u(y, cnt, min_cnt=10):
    """y: G theo bin hạng. Trả về (hệ số bậc 2, đỉnh, độ dốc đầu, độ dốc cuối, SSE tuyến-đơn-điệu-tăng, SSE bậc 2)."""
    from sklearn.isotonic import IsotonicRegression
    x = np.arange(len(y)); ok = np.isfinite(y) & (cnt >= min_cnt)
    if ok.sum() < 5:
        return (np.nan,) * 6
    xo, yo = x[ok].astype(float), y[ok]
    c2, c1, c0 = np.polyfit(xo, yo, 2)
    sse_q = float(np.square(yo - np.polyval([c2, c1, c0], xo)).sum())
    iso = IsotonicRegression(increasing=True).fit(xo, yo)
    sse_m = float(np.square(yo - iso.predict(xo)).sum())
    peak = -c1 / (2 * c2) if c2 != 0 else np.nan
    return c2, peak, c1 + 2 * c2 * xo[0], c1 + 2 * c2 * xo[-1], sse_m, sse_q

def test_S3(df):
    """S3: căn chỉnh theo hạng khái niệm là chữ U ngược (không đơn điệu tăng); đỉnh dịch ra xa khi capacity tăng."""
    s = df[df.grid.isin(["p3s2", "smoke", "s5"]) & (df.family == "decoder")]
    rows = []
    for r in s.itertuples():
        c = cv(r.rid)
        if c is None or "Grank" not in c:
            continue
        c2, pk, sl0, sl1, sm, sq = _inverted_u(c["Grank"][r.k_star], c["rank_counts"])
        rows.append(dict(rid=r.rid, d=r.d, L=r.L, ln_params=r.ln_params, curv=c2, peak_bin=pk, slope_start=sl0, slope_end=sl1, sse_mono=sm, sse_quad=sq))
    t = pd.DataFrame(rows).dropna()
    if len(t) < 5:
        return _res("S3", NO_DATA, "")
    p_curv = float(stats.ttest_1samp(t.curv, 0, alternative="less").pvalue)
    p_l = float(stats.ttest_1samp(t.slope_start, 0, alternative="greater").pvalue)
    p_r = float(stats.ttest_1samp(t.slope_end, 0, alternative="less").pvalue)
    p_mono = wilcoxon_p((t.sse_mono - t.sse_quad).values)
    inter = float(((t.peak_bin > 0) & (t.peak_bin < N_RANK_BINS - 1)).mean())
    rho = stats.spearmanr(t.ln_params, t.peak_bin).correlation if len(t) > 4 else np.nan
    ev = (f"độ cong<0 p={p_curv:.3g}; dốc đầu>0 p={p_l:.3g}; dốc cuối<0 p={p_r:.3g}; đơn điệu bị bác (SSE_mono>SSE_quad) p={p_mono:.3g}; "
          f"đỉnh nằm trong khoảng hạng: {inter:.0%}; Spearman(log B, vị trí đỉnh)={rho:.2f}")
    ok = p_curv < 0.05 and p_l < 0.05 and p_r < 0.05 and p_mono < 0.05 and inter >= 0.5
    return _res("S3", "ủng hộ" if ok else ("không ủng hộ" if p_l >= 0.5 else "chưa kết luận"), ev, t)

def test_S4(df):
    """S4: với dữ liệu bằng nhau, ngôn ngữ có fertility thấp nhất trở thành pivot (logit-lens ở tầng quanh waist); A3 dự đoán theo khối lượng dữ liệu."""
    s = sel(df, grid="s4")
    rows = []
    for r in s.itertuples():
        lens = _ld(r.rid, "lens")
        if lens is None:
            continue
        M = lens["M"]; k = r.k_star; ks = [x for x in (k - 1, k, k + 1) if 0 < x < M.shape[0] - 1] or [k]
        score = np.array([np.mean([M[kk, [b for b in range(r.L) if b != l], l].mean() for kk in ks]) for l in range(r.L)])
        rows.append(dict(rid=r.rid, seed=r.seed, share=("40%" if _has(r.share_X) else "đều"), fert="thấp" if r.len_scale_X < 1 else "cao", fert_X=r.fert_X,
                         score_X=float(score[0]), diff_X=float(score[0] - score[1:].mean()), X_is_pivot=bool(score.argmax() == 0)))
    t = pd.DataFrame(rows)
    if t.empty:
        return _res("S4", NO_DATA, "")
    piv = t.pivot_table(index="seed", columns=["share", "fert"], values="diff_X")
    ev, verdict = "", "chưa kết luận"
    if ("đều", "thấp") in piv.columns and ("đều", "cao") in piv.columns:
        d_cost = (piv[("đều", "thấp")] - piv[("đều", "cao")]).values
        p_cost = wilcoxon_p(d_cost)
        ev = f"dữ liệu đều: pivot-score của X (fertility thấp) - (fertility cao) = {np.nanmean(d_cost):.4f}, p={p_cost:.3g}"
        d_data = np.nan
        if ("40%", "cao") in piv.columns:
            d_data = (piv[("40%", "cao")] - piv[("đều", "cao")]).values
            p_data = wilcoxon_p(d_data)
            ev += f"; hiệu ứng khối lượng dữ liệu (40% vs đều, fertility cao) = {np.nanmean(d_data):.4f}, p={p_data:.3g}"
            if p_cost >= 0.05 and p_data < 0.05:
                verdict = "không ủng hộ"; ev += " -> pivot chỉ theo dữ liệu (A3)"
        if p_cost < 0.05 and np.nanmean(d_cost) > 0:
            verdict = "ủng hộ"
    return _res("S4", verdict, ev, t)

def test_S5(df):
    """S5: độ sâu tương đối ρ* tương quan dương với fertility trung bình (cùng họ, cùng capacity)."""
    s = sel(df, grid="s5")
    if len(s) < 8:
        return _res("S5", NO_DATA, "")
    x = s.fert_mean.values - s.groupby("seed").fert_mean.transform("mean").values        # khử hiệu ứng seed
    y = s.rho_star.values - s.groupby("seed").rho_star.transform("mean").values
    rs, ps = stats.spearmanr(s.fert_mean, s.rho_star); rk, pk = stats.kendalltau(s.fert_mean, s.rho_star)
    rs_w, ps_w = stats.spearmanr(x, y)
    ev = f"Spearman(φ, ρ*)={rs:.2f} (p hai phía {ps:.3g}); Kendall={rk:.2f} (p={pk:.3g}); trong-seed Spearman={rs_w:.2f} (p={ps_w:.3g})"
    ok = rs > 0 and ps / 2 < 0.05 and rk > 0 and pk / 2 < 0.05
    bad = rs <= 0 and ps / 2 < 0.05
    return _res("S5", "ủng hộ" if ok else ("không ủng hộ" if bad else "chưa kết luận"), ev, s[["rid", "len_scale", "seed", "fert_mean", "rho_star", "S_edge"]])

def test_S6(df):
    """S6 (một phần): với diffusion, đổi mức che không làm dịch k* (TOST |Δρ*| <= 0.10) mà chủ yếu đổi chiều cao / các tầng cuối."""
    s = df[(df.family == "diffusion") & df.get("rho_t0.25", pd.Series(dtype=float)).notna()] if "rho_t0.25" in df.columns else df.iloc[0:0]
    if s.empty:
        return _res("S6", NO_DATA, "")
    rows = []
    for t in MASK_LEVELS[1:]:
        dr = (s[f"rho_t{t:g}"] - s.rho_star).values
        dS = (s[f"S_edge_t{t:g}"] - s.S_edge).values
        dL = (s[f"G_last_t{t:g}"] - s.G_last).values
        rows.append(dict(t=t, n=len(s), mean_drho=float(np.mean(dr)), tost_p=tost_p(dr, RHO_MARGIN), mean_dS_edge=float(np.mean(dS)), p_dS_two_sided=wilcoxon_p(dS, alt="two-sided"),
                         mean_dG_last=float(np.mean(dL)), p_dGlast_two_sided=wilcoxon_p(dL, alt="two-sided")))
    tt = pd.DataFrame(rows)
    fr = np.nan
    try:
        cols = [s.rho_star.values] + [s[f"rho_t{t:g}"].values for t in MASK_LEVELS[1:]]
        fr = float(stats.friedmanchisquare(*cols).pvalue)
    except Exception:
        pass
    ev = f"TOST k* không đổi (biên {RHO_MARGIN}): p={tt.tost_p.round(3).tolist()}; Friedman qua t: p={fr:.3g}; ΔG_last theo t: {tt.mean_dG_last.round(3).tolist()}"
    return _res("S6", "ủng hộ" if (tt.tost_p < 0.05).all() else "chưa kết luận", ev + " (kiểm 'đáy không dịch cùng Pha III' đầy đủ cần so sánh Pha III riêng)", tt)

def run_all_tests(df):
    return [f(df) for f in (test_P1, test_P2, test_P3, test_S2, test_S3, test_S4, test_S5, test_S6)]

In [ ]:
# ============================================================
# Cell D4 — Style hình (giữ style Study 1: PDF, chữ serif, không tiêu đề, chú thích (a)(b) dưới trục, chú giải chung phía trên)
# ============================================================
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D

def setup_fonts():
    avail = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("Times New Roman", "Liberation Serif", "DejaVu Serif"):
        if name in avail:
            plt.rcParams["font.family"] = name; plt.rcParams["mathtext.fontset"] = "stix"
            return name
setup_fonts()
plt.rcParams.update({"pdf.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 9, "axes.labelsize": 9, "legend.fontsize": 8, "figure.dpi": 110})
C = dict(LM="#1f77b4", SEM="#d62728", SURF="#2ca02c", init="#7f7f7f", o0="#1f77b4", o100="#ff7f0e",
         decoder="#1f77b4", encoder="#9467bd", moe="#2ca02c", diffusion="#d62728")
LCOL = {2: "#1f77b4", 3: "#17becf", 4: "#ff7f0e", 6: "#8c564b", 8: "#2ca02c"}
KP = K_LAYERS
RHO = np.arange(KP + 1) / KP

def _stack(rows, key, variant="trained", norm_L=False):
    out = []
    for r in rows.itertuples():
        c = cv(r.rid, variant)
        if c is not None:
            v = np.asarray(c[key], float)
            out.append(v / math.log(r.L) if norm_L else v)
    return np.array(out)

def draw(ax, rows, key, color, ls="-", label=None, variant="trained", norm_L=False, mark_peak=False, mark_min=False):
    M = _stack(rows, key, variant, norm_L)
    if M.size == 0:
        return None
    m, lo, hi = tcrit_band(M)
    ax.plot(RHO, m, color=color, ls=ls, lw=1.6, label=label)
    ax.fill_between(RHO, lo, hi, color=color, alpha=0.15, lw=0)
    if mark_peak:
        ax.plot(RHO[int(np.nanargmax(m))], np.nanmax(m), "o", color=color, ms=5, mfc="white", mew=1.4)
    if mark_min:
        ax.plot(RHO[int(np.nanargmin(m))], np.nanmin(m), "v", color=color, ms=5, mfc="white", mew=1.4)
    return m

def subcap(ax, text):
    ax.text(0.5, -0.30, text, transform=ax.transAxes, ha="center", va="top", fontsize=9)

def legend_top(fig, handles, ncol=None, y=1.02):
    fig.legend(handles=handles, loc="upper center", ncol=ncol or len(handles), frameon=False, bbox_to_anchor=(0.5, y))

def save_fig(fig, name):
    p = f"{FIG_DIR}/{name}.pdf"
    fig.savefig(p, bbox_inches="tight"); plt.show(); plt.close(fig)
    print("đã lưu", p)

def _lines(entries):
    return [Line2D([0], [0], color=c, ls=ls, lw=1.6, label=lb) for c, ls, lb in entries]

def _xlab(ax):
    ax.set_xlabel(r"relative depth $\rho=k/K$"); ax.set_xlim(0, 1)

In [ ]:
# ============================================================
# Cell D5 — Hình (mỗi hình ứng với một giả thuyết)
# ============================================================
def fig_P1(df):
    fams = [f for f in FAMILIES if len(sel(df, family=f, level="M4", overlap=0, task=["LM", "SEM"]))]
    if not fams:
        return print("[P1] chưa có dữ liệu")
    fig, axes = plt.subplots(3, len(fams), figsize=(2.6 * len(fams) + 0.4, 6.4), squeeze=False)
    for j, fam in enumerate(fams):
        base = sel(df, family=fam, level="M4", overlap=0)
        for t, ls in (("LM", "-"), ("SEM", "-")):
            s = sel(base, task=t)
            draw(axes[0, j], s, "D", C[t], ls, mark_min=True)
            draw(axes[1, j], s, "G", C[t], ls, mark_peak=True)
            draw(axes[2, j], s, "U", C[t], ls, norm_L=True)
        s = sel(base, task="SEM")
        for i, k in enumerate(("D", "G", "U")):
            draw(axes[i, j], s, k, C["init"], "--", variant="init", norm_L=(k == "U"))
        axes[0, j].set_ylim(0, 2.2); axes[1, j].set_ylim(-0.05, 1.0); axes[2, j].set_ylim(-0.05, 1.05)
        for i in range(3):
            axes[i, j].set_xlim(0, 1)
            if i < 2: axes[i, j].set_xticklabels([])
        _xlab(axes[2, j]); subcap(axes[2, j], f"({'abcd'[j]}) {fam}")
    axes[0, 0].set_ylabel(r"separation $D_k$"); axes[1, 0].set_ylabel(r"alignment $\bar G_k$"); axes[2, 0].set_ylabel(r"$U_k/\log|\mathcal{L}|$")
    legend_top(fig, _lines([(C["LM"], "-", "LM"), (C["SEM"], "-", "SEM"), (C["init"], "--", "untrained (init)")]))
    fig.tight_layout(); save_fig(fig, "fig_P1_four_families")

def fig_A1(df):
    levels = [lv for lv in ("M0", "M2", "M4") if len(sel(df, family="decoder", overlap=100, level=lv))]
    if not levels:
        return print("[A1] chưa có dữ liệu (lưới balanced)")
    fig, axes = plt.subplots(2, len(levels), figsize=(3.0 * len(levels) + 0.4, 5.0), squeeze=False)
    for j, lv in enumerate(levels):
        for ov, col in ((0, C["o0"]), (100, C["o100"])):
            for t, ls in (("LM", "-"), ("SEM", "--")):
                s = sel(df, family="decoder", level=lv, overlap=ov, task=t)
                draw(axes[0, j], s, "G", col, ls, mark_peak=True); draw(axes[1, j], s, "U", col, ls, norm_L=True)
        axes[0, j].set_ylim(-0.05, 1.0); axes[1, j].set_ylim(-0.05, 1.05); axes[0, j].set_xticklabels([])
        _xlab(axes[1, j]); subcap(axes[1, j], f"({'abc'[j]}) level {lv}")
    axes[0, 0].set_ylabel(r"alignment $\bar G_k$"); axes[1, 0].set_ylabel(r"$U_k/\log|\mathcal{L}|$")
    legend_top(fig, _lines([(C["o0"], "-", "phoneme overlap 0%"), (C["o100"], "-", "overlap 100% (balanced interface)"),
                            ("k", "-", "LM"), ("k", "--", "SEM")]), ncol=4)
    fig.tight_layout(); save_fig(fig, "fig_A1_balanced_interface")

def fig_P2(df):
    fams = [f for f in ("decoder", "encoder") if len(sel(df, family=f, level="M4", overlap=0, grid="p1p2"))]
    if not fams:
        return print("[P2] chưa có dữ liệu")
    fig, axes = plt.subplots(1, len(fams) + 1, figsize=(3.0 * (len(fams) + 1) + 0.4, 3.2), squeeze=False); axes = axes[0]
    for j, fam in enumerate(fams):
        for t in TASKS:
            draw(axes[j], sel(df, family=fam, level="M4", overlap=0, grid="p1p2", task=t), "G", C[t], mark_peak=True)
        axes[j].set_ylim(-0.05, 1.0); _xlab(axes[j]); subcap(axes[j], f"({'ab'[j]}) {fam}, level M4"); axes[j].set_ylabel(r"alignment $\bar G_k$")
    ax = axes[-1]
    for fi, fam in enumerate(fams):
        for ti, t in enumerate(TASKS):
            s = sel(df, family=fam, level="M4", overlap=0, grid="p1p2", task=t)
            if len(s):
                m, h = s.w.mean(), (stats.t.ppf(0.975, len(s) - 1) * s.w.std(ddof=1) / math.sqrt(len(s)) if len(s) > 1 else 0)
                ax.errorbar(fi + (ti - 1) * 0.18, m, yerr=h, fmt="o", color=C[t], capsize=3)
    ax.set_xticks(range(len(fams))); ax.set_xticklabels(fams); ax.set_ylabel(r"waist width $w$"); subcap(ax, f"({'abc'[len(fams)]}) waist width by task")
    legend_top(fig, _lines([(C[t], "-", t) for t in TASKS])); fig.tight_layout(); save_fig(fig, "fig_P2_task_dependence")

def fig_P3(df):
    s = df[df.has_twins & df.grid.isin(["p3s2", "smoke"])]
    if s.empty:
        return print("[P3] chưa có dữ liệu sinh đôi")
    def dA_curve(rows):
        M = [np.nanmean(x, 0) for x in (_delta_A(r) for r in rows.itertuples()) if x is not None]
        return np.array(M)
    fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2))
    d0 = 128 if (s.d == 128).any() else int(s.d.min())
    for L in sorted(s.L.unique()):
        M = dA_curve(s[(s.d == d0) & (s.L == L)])
        if M.size:
            m, lo, hi = tcrit_band(M); axes[0].plot(RHO, m, color=LCOL.get(L), lw=1.6, label=f"|L|={L}"); axes[0].fill_between(RHO, lo, hi, color=LCOL.get(L), alpha=0.15, lw=0)
    Lm = 4 if (s.L == 4).any() else int(s.L.min())
    for d, col in zip(sorted(s.d.unique()), plt.cm.viridis(np.linspace(0, 0.9, s.d.nunique()))):
        M = dA_curve(s[(s.d == d) & (s.L == Lm)])
        if M.size:
            m, lo, hi = tcrit_band(M); axes[1].plot(RHO, m, color=col, lw=1.6, label=f"d={d}"); axes[1].fill_between(RHO, lo, hi, color=col, alpha=0.15, lw=0)
    t = test_P3(df)["table"]
    if t is not None and len(t):
        for L in sorted(t.L.unique()):
            g = t[t.L == L].groupby("d").agg(x=("ln_params", "mean"), m=("dA_kstar", "mean"), sd=("dA_kstar", "std")).reset_index()
            axes[2].errorbar(g.x, g.m, yerr=g.sd.fillna(0), fmt="-o", color=LCOL.get(L), capsize=3, ms=4, label=f"|L|={L}")
    for ax in axes[:2]:
        ax.axhline(0, color="k", lw=0.8); _xlab(ax); ax.set_ylabel(r"$\Delta A=A_{joint}-A_{indep}$"); ax.legend(frameon=False)
    axes[2].axhline(0, color="k", lw=0.8); axes[2].set_xlabel(r"$\ln B$ (non-embedding params)"); axes[2].set_ylabel(r"$\Delta A$ at waist"); axes[2].legend(frameon=False)
    subcap(axes[0], f"(a) by |L| (d={d0})"); subcap(axes[1], f"(b) by capacity (|L|={Lm})"); subcap(axes[2], "(c) ΔA at waist vs capacity")
    fig.tight_layout(); save_fig(fig, "fig_P3_joint_vs_twins")

def fig_S2(df):
    s = sel(df, grid="p3s2", family="decoder")
    if s.empty:
        return print("[S2] chưa có dữ liệu")
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.2))
    for L in sorted(s.L.unique()):
        x = s[s.L == L]; g = x.groupby("d").G_at_kstar.agg(["mean", "std", "count"]).reset_index()
        axes[0].plot(np.log2(x.d), x.G_at_kstar, ".", color=LCOL.get(L), alpha=0.25, ms=4)
        axes[0].plot(np.log2(g.d), g["mean"], "-o", color=LCOL.get(L), lw=1.6, ms=4, label=f"|L|={L}")
        axes[1].plot(np.log2(g.d), g["std"], "-s", color=LCOL.get(L), lw=1.6, ms=4, label=f"|L|={L}")
    for ax in axes:
        ax.set_xlabel(r"$\log_2 d$ (capacity)"); ax.legend(frameon=False)
    axes[0].set_ylabel(r"$\bar G_{k^*}$ at waist"); axes[1].set_ylabel(r"SD across seeds")
    subcap(axes[0], "(a) alignment at the waist"); subcap(axes[1], "(b) seed variance")
    fig.tight_layout(); save_fig(fig, "fig_S2_staircase")

def fig_S3(df):
    s = df[df.grid.isin(["p3s2", "smoke"]) & (df.family == "decoder")]
    if s.empty:
        return print("[S3] chưa có dữ liệu")
    Lm = 4 if (s.L == 4).any() else int(s.L.min()); s4 = s[s.L == Lm]
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.2)); xs = np.arange(N_RANK_BINS)
    for d, col in zip(sorted(s4.d.unique()), plt.cm.viridis(np.linspace(0, 0.9, s4.d.nunique()))):
        Ms = [cv(r.rid)["Grank"][r.k_star] for r in s4[s4.d == d].itertuples() if cv(r.rid) is not None and "Grank" in cv(r.rid)]
        if Ms:
            m, lo, hi = tcrit_band(np.array(Ms)); axes[0].plot(xs, m, "-o", color=col, ms=3, lw=1.5, label=f"d={d}"); axes[0].fill_between(xs, lo, hi, color=col, alpha=0.12, lw=0)
    axes[0].set_xlabel(r"concept frequency rank $\lfloor\log_2 r\rfloor$ (0 = most frequent)"); axes[0].set_ylabel(r"alignment $G$ at the waist"); axes[0].legend(frameon=False, ncol=2)
    t = test_S3(df)["table"]
    if t is not None and len(t):
        sc = axes[1].scatter(t.ln_params, t.peak_bin, c=t.L, cmap="viridis", s=14)
        axes[1].set_xlabel(r"$\ln B$"); axes[1].set_ylabel("rank bin of the peak"); fig.colorbar(sc, ax=axes[1], label="|L|")
    subcap(axes[0], "(a) alignment vs concept rank"); subcap(axes[1], "(b) peak position vs capacity")
    fig.tight_layout(); save_fig(fig, "fig_S3_frequency_inverted_U")

def fig_S4(df):
    t = test_S4(df)["table"]
    if t is None or t.empty:
        return print("[S4] chưa có dữ liệu")
    conds = [(s_, f_) for s_ in ("đều", "40%") for f_ in ("thấp", "cao")]
    names = {"đều": "equal data", "40%": "X: 40% data", "thấp": "low fertility", "cao": "high fertility"}
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3))
    for i, (s_, f_) in enumerate(conds):
        v = t[(t.share == s_) & (t.fert == f_)].diff_X.values
        if len(v):
            axes[0].bar(i, v.mean(), color="#9ecae1" if f_ == "thấp" else "#fdae6b", edgecolor="k", lw=0.6)
            axes[0].plot(np.full(len(v), i) + np.random.default_rng(0).uniform(-0.12, 0.12, len(v)), v, "k.", ms=4)
    axes[0].axhline(0, color="k", lw=0.8); axes[0].set_xticks(range(4)); axes[0].set_xticklabels([f"{names[a]}\n{names[b]}" for a, b in conds], fontsize=7)
    axes[0].set_ylabel("pivot score of X minus mean of others")
    sub = sel(df, grid="s4"); cols = {}
    for (s_, f_), col in zip(conds, ["#08519c", "#6baed6", "#a63603", "#fd8d3c"]):
        ys = []
        for r in sub.itertuples():
            if ("40%" if _has(r.share_X) else "đều") == s_ and ("thấp" if r.len_scale_X < 1 else "cao") == f_:
                ln = _ld(r.rid, "lens")
                if ln is not None:
                    M = ln["M"]; ys.append([M[k, 1:, 0].mean() for k in range(M.shape[0])])
        if ys:
            m, lo, hi = tcrit_band(np.array(ys)); axes[1].plot(RHO, m, color=col, lw=1.6, label=f"{names[s_]}, {names[f_]}"); axes[1].fill_between(RHO, lo, hi, color=col, alpha=0.12, lw=0)
    _xlab(axes[1]); axes[1].set_ylabel("logit-lens mass on X tokens (input in other languages)"); axes[1].legend(frameon=False, fontsize=7)
    subcap(axes[0], "(a) pivot score at the waist"); subcap(axes[1], "(b) X-mass across depth")
    fig.tight_layout(); save_fig(fig, "fig_S4_cost_driven_pivot")

def fig_S5(df):
    s = sel(df, grid="s5")
    if s.empty:
        return print("[S5] chưa có dữ liệu")
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.2)); lss = sorted(s.len_scale.unique())
    sc = axes[0].scatter(s.fert_mean, s.rho_star, c=s.len_scale, cmap="viridis", s=18)
    if len(s) > 3:
        b, a = np.polyfit(s.fert_mean, s.rho_star, 1); xx = np.linspace(s.fert_mean.min(), s.fert_mean.max(), 20); axes[0].plot(xx, a + b * xx, "k--", lw=1)
    axes[0].set_xlabel(r"mean tokens per word $\phi$"); axes[0].set_ylabel(r"waist depth $\rho^*$"); fig.colorbar(sc, ax=axes[0], label="len_scale")
    for ls_, col in zip(lss, plt.cm.viridis(np.linspace(0, 0.9, len(lss)))):
        draw(axes[1], s[s.len_scale == ls_], "G", col, label=f"len_scale={ls_:g}", mark_peak=True)
    _xlab(axes[1]); axes[1].set_ylabel(r"alignment $\bar G_k$"); axes[1].legend(frameon=False, fontsize=7)
    subcap(axes[0], r"(a) $\rho^*$ vs fertility"); subcap(axes[1], "(b) alignment curves")
    fig.tight_layout(); save_fig(fig, "fig_S5_tokenizer_depth")

def fig_S6(df):
    s = sel(df, family="diffusion", level="M4", overlap=0)
    if s.empty or "rho_t0.25" not in df.columns:
        return print("[S6] chưa có dữ liệu diffusion")
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.2)); tl = MASK_LEVELS
    for t, col in zip(tl, plt.cm.plasma(np.linspace(0.1, 0.8, len(tl)))):
        draw(axes[0], s[s.task == "SEM"] if (s.task == "SEM").any() else s, "G", col, variant="trained" if t == 0 else f"t{t:g}", label=f"t={t:g}", mark_peak=True)
    _xlab(axes[0]); axes[0].set_ylabel(r"alignment $\bar G_k$"); axes[0].legend(frameon=False)
    for task in TASKS:
        x = s[s.task == task]
        if len(x):
            ys = np.stack([x.rho_star.values] + [x[f"rho_t{t:g}"].values for t in tl[1:]], 1)
            m, lo, hi = tcrit_band(ys); axes[1].errorbar(tl, m, yerr=(m - lo), fmt="-o", color=C[task], capsize=3, ms=4, label=task)
    axes[1].axhspan(-RHO_MARGIN, RHO_MARGIN, alpha=0.0); axes[1].set_xlabel("mask level t"); axes[1].set_ylabel(r"waist depth $\rho^*$"); axes[1].legend(frameon=False)
    subcap(axes[0], "(a) alignment by mask level"); subcap(axes[1], r"(b) $\rho^*$ vs mask level")
    fig.tight_layout(); save_fig(fig, "fig_S6_diffusion_mask")

def fig_forest(df):
    s = sel(df, overlap=0, grid=["p1p2", "moe_diff"])
    if s.empty:
        return print("[forest] chưa có dữ liệu")
    g = s.groupby(["family", "level", "task"]).S_edge.agg(["mean", "std", "count"]).reset_index()
    g["h"] = [stats.t.ppf(0.975, c - 1) * sd / math.sqrt(c) if c > 1 else 0 for sd, c in zip(g["std"].fillna(0), g["count"])]
    g = g.sort_values(["family", "level", "task"]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(4.8, 0.22 * len(g) + 1.2))
    for i, r in g.iterrows():
        ax.errorbar(r["mean"], i, xerr=r.h, fmt="o", color=C[r.family], capsize=2, ms=4)
    ax.axvline(WAIST_S_FLOOR, color="k", ls=":", lw=1); ax.set_yticks(range(len(g))); ax.set_yticklabels([f"{r.family} {r.level} {r.task}" for r in g.itertuples()], fontsize=7)
    ax.set_xlabel(r"edge depth $S_{edge}$ (mean, CI95 over seeds)"); ax.invert_yaxis(); fig.tight_layout(); save_fig(fig, "fig_forest_edge_depth")

def make_all_figures(df):
    for f in (fig_P1, fig_A1, fig_P2, fig_P3, fig_S2, fig_S3, fig_S4, fig_S5, fig_S6, fig_forest):
        try:
            f(df)
        except Exception as e:
            print(f"[{f.__name__}] lỗi: {repr(e)[:200]}"); traceback.print_exc()

In [ ]:
# ============================================================
# Cell D6 — Báo cáo: bảng kết luận từng giả thuyết + số liệu chính
# ============================================================
def build_report(df, results):
    L_ = []; w = L_.append
    w(f"# Báo cáo Study 2a — ngôn ngữ tổng hợp ({datetime.now():%Y-%m-%d %H:%M})\n")
    w("> Mọi kết luận chỉ ở **quy mô nhỏ, trong ngôn ngữ tổng hợp** (≤ 14M tham số, K = 8). Kết quả âm chưa bác bỏ LE-D ở quy mô lớn; "
      "P4 (dự báo ngoài mẫu) thuộc Study 5, ở đây chỉ xuất bảng đầu vào.\n")
    cov = df.groupby("grid").size().to_dict() if len(df) else {}
    w("## Phạm vi dữ liệu\n"); w("| lưới | run hoàn thành |\n|---|---|")
    for g_, n_ in cov.items():
        w(f"| {g_} | {n_} |")
    w("\n## Kết luận theo giả thuyết\n"); w("| giả thuyết | kết luận | bằng chứng |\n|---|---|---|")
    for r in results:
        w(f"| **{r['hyp']}** | {r['verdict']} | {r['evidence']} |")
    if len(df):
        w("\n## Waist theo (họ, mức, tác vụ), overlap 0%\n")
        g = sel(df, overlap=0, grid=["p1p2", "moe_diff", "gate_g2", "smoke"]).groupby(["family", "level", "task"]).agg(
            n=("rid", "size"), frac_waist=("has_waist", "mean"), rho=("rho_star", "median"), S_edge=("S_edge", "median"), width=("w", "median"),
            U0=("U0_over_logL", "median"), U_waist=("U_at_kstar_over_logL", "median")).round(3).reset_index()
        try:
            w(g.to_markdown(index=False))
        except Exception:                                   # chưa cài tabulate
            w("```\n" + g.to_string(index=False) + "\n```")
        if (df.U0_over_logL < 0.9).any():
            w("\n> ⚠ Có run với U₀/log|L| < 0.9 ở tầng embedding: ngôn ngữ chưa đọc ra hoàn toàn từ chữ (kiểm tra giao diện/overlap).")
    w("\n## Hình\n")
    for f_ in sorted(os.listdir(FIG_DIR)):
        w(f"- `figures/{f_}`")
    w("\n## Lưu ý đọc kết quả\n- Waist của từng run theo tiêu chí Study 1 (đỉnh G trong thân, CI dưới của edge depth > 0.05, CKA khớp, vượt `init`).\n"
      "- Hình hourglass lý tưởng của Corollary 1 (quay lại H(Λ) ở tầng cuối) hiếm khi đạt: so G_K với G_0 trong bảng đặc trưng.\n"
      "- S6 mới kiểm k* không dịch khi đổi mức che; việc tách riêng Pha III cần thêm phép đo ở các tầng cuối.\n")
    txt = "\n".join(L_)
    with open(f"{ANALYSIS_DIR}/report_study2a.md", "w", encoding="utf-8") as f:
        f.write(txt)
    pd.DataFrame([dict(hypothesis=r["hyp"], verdict=r["verdict"], evidence=r["evidence"]) for r in results]).to_csv(f"{ANALYSIS_DIR}/hypotheses_summary.csv", index=False)
    for r in results:
        if r["table"] is not None and len(r["table"]):
            r["table"].to_csv(f"{ANALYSIS_DIR}/table_{r['hyp']}.csv", index=False)
    return txt

def export_p4_inputs(df):
    """Đầu vào cho Study 5 (không kiểm định ở đây): ρ*, w, ΔA, fertility của các cấu hình giữ ra và của lưới huấn luyện."""
    rows = []
    for r in df[df.grid.isin(["p3s2", "heldout_p4", "s5", "gate_g2", "smoke"])].itertuples():
        dA = _delta_A(r) if r.has_twins else None
        rows.append(dict(rid=r.rid, grid=r.grid, L=r.L, d=r.d, ln_params=r.ln_params, len_scale=r.len_scale, seed=r.seed, fert_mean=r.fert_mean,
                         rho_star=r.rho_star, w=r.w, S_edge=r.S_edge, dA_kstar=float(np.nanmean(dA[:, r.k_star])) if dA is not None else np.nan))
    out = pd.DataFrame(rows)
    out.to_csv(f"{ANALYSIS_DIR}/p4_inputs_for_study5.csv", index=False)
    return out

def regenerate_all():
    df = build_feature_table()
    if df.empty:
        print("Chưa có run nào hoàn thành trong", RUN_DIR); return None, []
    res = run_all_tests(df)
    export_p4_inputs(df)
    make_all_figures(df)
    txt = build_report(df, res)
    print(txt)
    return df, res

In [ ]:
# ============================================================
# Cell D7 — CHẠY phân tích: đặc trưng -> kiểm định -> hình -> báo cáo (tất cả lưu ở STUDY_DIR/analysis, STUDY_DIR/figures)
# ============================================================
df_feat, results = regenerate_all()